<a href="https://colab.research.google.com/github/netsetos/agentic-ai-weekend-gcp-learners/blob/rag-production-hardening/module-12-production-deploy/lesson-12.5-ingestion/notebooks/GCP_Capstone_12.5_Ingestion.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 12.5 Productionize Ingestion — Three Objects Into the Lane
**Netsetos GenAI Engineering — GCP Capstone** · Module 12 · rebuilt on the live lane, 10 September 2026

The one service nothing deployed until this lesson is the one that has been running the longest: `documind-ingest` indexed the fourteen real documents of every tenant on 6 September, and Module 9's figures and video segments after that. This notebook is where it comes from - the heredocs at the end are the kit's source - and the story now runs against it. A markdown note with a fact nobody else has goes in, the worker's one log line is waited for, the fact comes back through the one `retrieve()`. The same bytes go in again and are refused by the claim. A zero-byte object goes in and is refused at the edge with a 400, and the dead-letter subscription is peeked. A PNG goes in and becomes a figure chunk. The contracts, the claim race and the poison path stay as the theory they are, exercised against the lane's own modules.


## Setup


In [ ]:
!pip install -q google-genai==2.22.0 google-cloud-storage==3.13.1 google-cloud-firestore==2.30.0 pydantic==2.13.5 requests==2.34.2

from google.colab import auth
auth.authenticate_user()

PROJECT_ID = "documind-ai-YOUR-ID"   # CHANGE THIS: the project the lane runs in (make up, lesson 4.8)
REGION     = "us-central1"
TENANT     = "acme"
KIT        = "/content/agentic-ai-weekend-gcp-learners"   # the kit: deploy/shared is the tool layer every lesson on the lane imports
BRANCH     = "main"        # the learner repo's branch: the notebooks and the kit (deploy/) ship there together

import os, subprocess, sys
import google.auth
from google.auth.transport.requests import AuthorizedSession
from google import genai
from google.genai import types

if not os.path.isdir(KIT):
    subprocess.run(["git", "clone", "--depth", "1", "-q", "-b", BRANCH,
                    "https://github.com/netsetos/agentic-ai-weekend-gcp-learners", KIT], check=True)
sys.path.insert(0, f"{KIT}/deploy")                   # `from shared import ...` - the same layer every service imports

# The lane's URLs are deterministic: service name + project NUMBER (eventarc.tf builds them the same way).
creds, _ = google.auth.default()
NUMBER = AuthorizedSession(creds).get(
    f"https://cloudresourcemanager.googleapis.com/v1/projects/{PROJECT_ID}").json()["projectNumber"]
API_URL       = f"https://documind-api-{NUMBER}.{REGION}.run.app"
UPLOAD_BUCKET = f"{PROJECT_ID}-uploads"     # storage.tf: the bucket eventarc.tf watches - the corpus, media included
MEDIA_BUCKET  = f"{PROJECT_ID}-media"       # storage.tf: generated assets, 30-day lifecycle (a cache, not a record)
DATASETS      = f"{PROJECT_ID}-datasets"    # storage.tf (Module 10): the frozen tuning dataset and 10.5's GGUF
GATEWAY_URL   = f"https://documind-gateway-{NUMBER}.{REGION}.run.app"   # 11.3: LiteLLM on Cloud Run, behind IAM (make deploy-gateway)
SLM_URL       = f"https://documind-slm-{NUMBER}.{REGION}.run.app"       # 11.4: Ollama on an L4, min-instances 0 (make deploy-slm)
os.environ.update({
    "GOOGLE_CLOUD_PROJECT": PROJECT_ID,
    "GOOGLE_CLOUD_LOCATION": "global",              # Gemini 3.x generation is served from the global endpoint
    "GOOGLE_GENAI_USE_VERTEXAI": "TRUE",
    "DOCUMIND_PROFILE": "gcp",
    "RAG_API_URL": API_URL,
    "RAG_TIMEOUT_S": "90",                          # 7.2's finding: a cold API takes longer than the default 20 s
    # A notebook has no metadata server to be anyone with: the kit mints its ID tokens AS this roster
    # member (7.1). On Cloud Run the service's own account is the identity and nothing is set.
    "DOCUMIND_IMPERSONATE_SA": f"documind-ui-sa@{PROJECT_ID}.iam.gserviceaccount.com",
})
MEMBER_SA   = os.environ["DOCUMIND_IMPERSONATE_SA"]
OUTSIDER_SA = f"documind-outsider-sa@{PROJECT_ID}.iam.gserviceaccount.com"   # IAM admits it, no roster does (4.8, 7.2)

from shared import documind_tools    # THE one retrieve(). Imported, never pasted - the contract gate fails a paste.
gen = genai.Client(enterprise=True, project=PROJECT_ID, location="global")   # every generate_content in this lesson
sys.path.insert(0, f"{KIT}/deploy/services/ingest")   # contracts, idempotency: the worker's own modules, imported from the clone
from google.cloud import firestore
db = firestore.Client(project=PROJECT_ID)
import datetime
STAMP = datetime.datetime.now().strftime("%Y%m%d%H%M%S")   # this run's objects and fact are unique

print("kit:", KIT, "| API:", API_URL, "| gateway:", GATEWAY_URL, "| slm:", SLM_URL)


## Cell 1: The API, the gateway and the SLM, called the way the lane calls them


In [ ]:
import json, requests, time, subprocess, datetime
from google.cloud import storage

# THE API, THE GATEWAY AND THE SLM, CALLED THE WAY THE LANE CALLS THEM: one ID token per request, minted AS the roster
# member for the service's own URL (the kit mints it: documind_tools._id_token). Every service on the lane is behind
# Cloud Run IAM; there is no master key and no API key to paste.
def api(path: str, body: dict | None = None, base: str | None = None, timeout: int = 120, token: str | None = "member") -> tuple[int, dict | str]:
    """POST one API route (or a candidate revision's, with base=) - as documind-ui-sa by default, with token=None as
    nobody, or with a token minted as another account. Returns (status, json-or-text)."""
    url = (base or API_URL).rstrip("/")
    headers = {}
    if token == "member":
        headers["Authorization"] = f"Bearer {documind_tools._id_token(API_URL)}"     # the audience is the canonical URL
    elif token:
        headers["Authorization"] = f"Bearer {token}"
    r = requests.post(f"{url}{path}", json=body, headers=headers, timeout=timeout)
    try:
        return r.status_code, r.json()
    except ValueError:
        return r.status_code, r.text[:400]

def api_get(path: str, base: str | None = None, timeout: int = 60) -> tuple[int, dict | str]:
    """GET one API route as the roster member: /version, /health."""
    url = (base or API_URL).rstrip("/")
    r = requests.get(f"{url}{path}", headers={"Authorization": f"Bearer {documind_tools._id_token(API_URL)}"}, timeout=timeout)
    try:
        return r.status_code, r.json()
    except ValueError:
        return r.status_code, r.text[:400]

def gateway(model: str, content: str, json_mode: bool = False, system: str | None = None, max_tokens: int = 200,
            timeout: int = 150) -> tuple[int, dict | str, dict]:
    """One OpenAI-compatible completion through the gateway (11.3), as the roster member. Returns (status, body, headers)."""
    msgs = ([{"role": "system", "content": system}] if system else []) + [{"role": "user", "content": content}]
    body = {"model": model, "messages": msgs, "max_tokens": max_tokens}
    if json_mode:
        body["response_format"] = {"type": "json_object"}
    r = requests.post(f"{GATEWAY_URL}/v1/chat/completions", json=body, timeout=timeout,
                      headers={"Authorization": f"Bearer {documind_tools._id_token(GATEWAY_URL)}"})
    try:
        return r.status_code, r.json(), dict(r.headers)
    except ValueError:
        return r.status_code, r.text[:400], dict(r.headers)

def service(name: str, region: str | None = None) -> dict:
    """A Cloud Run service as deployed: its env, labels, traffic, image and floor - read with gcloud, the way 10.3 did."""
    r = subprocess.run(["gcloud", "run", "services", "describe", name, "--region", region or REGION, "--project", PROJECT_ID,
                        "--format=json"], capture_output=True, text=True)
    if r.returncode != 0:
        return {}
    j = json.loads(r.stdout)
    c = j["spec"]["template"]["spec"]["containers"][0]
    return {"env": {e["name"]: e.get("value", "") for e in c.get("env", [])}, "image": c.get("image"),
            "labels": (j["metadata"].get("labels") or {}), "traffic": j.get("status", {}).get("traffic", []),
            "url": j.get("status", {}).get("url"), "sa": j["spec"]["template"]["spec"].get("serviceAccountName"),
            "annotations": (j["spec"]["template"]["metadata"].get("annotations") or {}),
            "min_instances": (j["spec"]["template"]["metadata"].get("annotations") or {}).get("autoscaling.knative.dev/minScale", "0")}

# The usage rows the API logs - the ONE shape every observability consumer reads (12.3, tenant_daily). On the lean
# lane they live in Cloud Logging; this reads the last few for a surface, newest first.
def usage_rows(minutes: int = 15, limit: int = 20, event: str = "query", service_name: str = "documind-api") -> list[dict]:
    since = (datetime.datetime.now(datetime.timezone.utc) - datetime.timedelta(minutes=minutes)).strftime("%Y-%m-%dT%H:%M:%SZ")
    r = subprocess.run(["gcloud", "logging", "read",
                        f'resource.type="cloud_run_revision" AND resource.labels.service_name="{service_name}" '
                        f'AND jsonPayload.event="{event}" AND timestamp>="{since}"',
                        "--project", PROJECT_ID, "--limit", str(limit), "--format=json"], capture_output=True, text=True)
    try:
        return [e["jsonPayload"] for e in json.loads(r.stdout or "[]")]
    except ValueError:
        return []

# THE TWENTY LINES THAT MATTER, FROM THE CLONE. Module 12's notebooks are where the kit's files come from (the heredoc
# cells at the end of each notebook are what extract_documind.py reads), so the walls stay there and the story reads
# the file the lane actually runs, around one line, with the file's length beside it.
def excerpt(rel: str, needle: str, before: int = 0, after: int = 14) -> str:
    lines = open(f"{KIT}/deploy/{rel}", encoding="utf-8").read().splitlines()
    i = next(n for n, l in enumerate(lines) if needle in l)
    lo, hi = max(0, i - before), min(len(lines), i + after)
    return f"# {rel}:{lo + 1}-{hi}  ({len(lines)} lines)\n" + "\n".join(lines[lo:hi])

def gcloud(*args: str) -> str:
    """One gcloud read, as the notebook's account, stdout only."""
    r = subprocess.run(["gcloud", *args, "--project", PROJECT_ID], capture_output=True, text=True)
    return r.stdout.strip() if r.returncode == 0 else f"(gcloud: {r.stderr.strip()[:200]})"

gcs = storage.Client(project=PROJECT_ID)

sys.path.insert(0, f"{KIT}/deploy/evals")               # run_eval, usage_rows, judge
sys.path.insert(0, f"{KIT}/deploy/services/rag-api")    # the API's own modules, for the excerpts and the pure functions
print("helpers: api(), api_get(), gateway(), service(), usage_rows(), excerpt(), gcloud(); the kit's evals/ and rag-api/ on sys.path")


## Cell 2: The lane, watched
A helper that waits for the worker's log line, and one that puts an object under the tenant's prefix.


In [ ]:
from contracts import IngestMessage, DocumentContract, sha256_of
from google.cloud.firestore_v1.base_query import FieldFilter

# THE LANE, WATCHED. An object under gs://PROJECT-uploads/<tenant>/ is the whole API of ingestion: Eventarc turns the
# finalize event into a Pub/Sub push to documind-ingest, the worker claims the content hash, parses, scans, chunks,
# embeds, indexes, and logs ONE line - ingest_ok with the tenant, the doc_key, the chunk and page counts. This helper
# waits for that line the way make ingest-one does, and the next cells upload three very different objects.
def wait_for(event: str, since: datetime.datetime, tenant: str | None = TENANT, minutes: int = 5) -> dict | None:
    """The worker's log line for this upload, or None: polls Cloud Logging every 10 s, up to `minutes`."""
    q = (f'resource.type="cloud_run_revision" AND resource.labels.service_name="documind-ingest" AND jsonPayload.event="{event}" '
         f'AND timestamp>="{since.strftime("%Y-%m-%dT%H:%M:%SZ")}"' + (f' AND jsonPayload.tenant="{tenant}"' if tenant else ""))
    for _ in range(minutes * 6):
        r = subprocess.run(["gcloud", "logging", "read", q, "--project", PROJECT_ID, "--limit", "1", "--format=json"], capture_output=True, text=True)
        rows = json.loads(r.stdout or "[]")
        if rows:
            return rows[0]["jsonPayload"]
        time.sleep(10)
    return None

def chunks_of(source_uri: str) -> list[dict]:
    """The chunks the worker wrote for one object - the two equality filters need no composite index."""
    q = (db.collection("chunks").where(filter=FieldFilter("tenant_id", "==", TENANT))
           .where(filter=FieldFilter("source_uri", "==", source_uri)))
    return [d.to_dict() for d in q.stream()]

def upload(name: str, data: bytes, content_type: str) -> str:
    """One object under the tenant's prefix - the same gcloud storage cp the Makefile runs, as a client call."""
    blob = gcs.bucket(UPLOAD_BUCKET).blob(f"{TENANT}/{name}")
    blob.upload_from_string(data, content_type=content_type)
    return f"gs://{UPLOAD_BUCKET}/{TENANT}/{name}"

print("wait_for(event, since), upload(name, bytes, content_type) - the worker's contract:", excerpt("services/ingest/main.py", "def push(", 1, 3).splitlines()[0])


## Cell 3: A new document with a fact nobody else has


In [ ]:
# A NEW DOCUMENT WITH A FACT NOBODY ELSE HAS. Markdown is parsed directly (text/* skips Document AI), so this is the
# cheapest object the lane ingests - and the fact is unique to this run, so retrieving it afterwards proves the LANE
# indexed it, not that a demo corpus already had it. The doc_key is computed here the way the worker computes it
# (contracts.py: tenant + sha256 of the CONTENT), so the Firestore claim can be read back by name.
FACT_CODE = f"RN-{STAMP[-6:]}"
NOTE = f"""# Module 12 rehearsal note {STAMP}

The rehearsal reference code for this ingestion run is {FACT_CODE}. Quote it exactly when asked for the rehearsal reference code.

The note was uploaded from lesson 12.5 to show the lane ingesting a document end to end: the object landed in the uploads
bucket, Eventarc delivered it, the worker claimed its content hash, chunked it, scanned it, embedded it and indexed it.
""".encode()
since = datetime.datetime.now(datetime.timezone.utc)
uri = upload(f"module12_rehearsal_note_{STAMP}.md", NOTE, "text/markdown")
doc = DocumentContract(tenant_id=TENANT, sha256=sha256_of(NOTE), gcs_uri=uri, pages=0)
print("uploaded", uri, "| doc_key", doc.doc_key[:40], "...")
line = wait_for("ingest_ok", since)
assert line, "no ingest_ok in 5 min: gcloud logging read ... service_name=documind-ingest (is the push subscription alive? eventarc.tf)"
print("the worker's line:", {k: line.get(k) for k in ("event", "tenant", "doc_key", "chunks", "pages", "kinds")})
assert line["doc_key"] == doc.doc_key, "the worker's key is the content hash this notebook computed"
claim = db.collection("documents").document(doc.doc_key).get().to_dict()
print("the claim   :", {k: claim.get(k) for k in ("status", "chunks", "gcs_uri")})
assert claim["status"] == "indexed"


### Retrieved, through the one retrieve()


In [ ]:
# RETRIEVED, THROUGH THE ONE retrieve(). The fact is in the index a few seconds after the line; the kit's retrieve()
# (Modules 6-8: the single entry point every brain and both agents call) asks the API, and the API cites the chunk
# the worker wrote - kind text, the object's URI as the source. Nothing about this document was special-cased.
hit = None
for attempt in range(6):
    res = documind_tools.retrieve("What is the rehearsal reference code for this ingestion run?", TENANT, top_k=5)
    hit = next((c for c in res.get("citations", []) if FACT_CODE in (c.get("quote") or c.get("text") or "")), None)
    if hit or FACT_CODE in (res.get("answer") or ""):
        break
    time.sleep(10)
print("answerable:", res.get("answerable"), "| answer:", (res.get("answer") or "")[:120])
print("cited     :", [(c.get("source") or c.get("source_uri", "")).split("/")[-1] for c in res.get("citations", [])][:3])
assert FACT_CODE in (res.get("answer") or "") or hit, "the fact uploaded three cells ago must come back through the door"


## Cell 4: The same bytes, again
> **The idempotency key is a hash of the content, never the Pub/Sub message id.** The message id is stable across redelivery, so it would deduplicate a retry - but the same file uploaded twice is two messages with two ids, and that is the duplicate users actually create.


In [ ]:
# THE SAME BYTES, AGAIN. A second upload is a second Pub/Sub message with a second id - the message id would not
# catch it - but the same content hash, and claim() refuses it: the worker returns 200 {"status": "duplicate"}, which
# ACKS the message (a success, not a failure). Nothing is re-parsed, re-embedded or re-billed, and the chunk ids being
# deterministic (tenant:sha256#i) means even a forced re-run would UPSERT, never add a second copy.
before = len(chunks_of(uri))
since = datetime.datetime.now(datetime.timezone.utc)
uri2 = upload(f"module12_rehearsal_note_{STAMP}_again.md", NOTE, "text/markdown")
print("uploaded the same bytes as", uri2.split("/")[-1])
time.sleep(45)
again = wait_for("ingest_ok", since, minutes=1)
after = len(chunks_of(uri))
claim = db.collection("documents").document(doc.doc_key).get().to_dict()
print(f"chunks under the first object: {before} -> {after} | a second ingest_ok: {bool(again)} | claim status: {claim.get('status')} gcs_uri: {claim.get('gcs_uri', '').split('/')[-1]}")
assert not again and after == before, "the duplicate must be acked without indexing"
print("the claim still names the FIRST object: the second upload was answered from the claim, not the index")


### The claim, as a race
The transaction in `idempotency.py` is what makes the cell above deterministic. Against a fake Firestore, so the sequence is visible without the plumbing:


In [ ]:
# claim() against a fake Firestore, so the RACE is visible without a project.
class FakeDoc:
    def __init__(self, store, key): self.store, self.key = store, key
    @property
    def exists(self): return self.key in self.store

class FakeRef:
    def __init__(self, store, key): self.store, self.key = store, key
    def get(self, transaction=None): return FakeDoc(self.store, self.key)
    def set(self, data, merge=False):
        self.store.setdefault(self.key, {}).update(data) if merge else \
            self.store.__setitem__(self.key, dict(data))

class FakeDB:
    def __init__(self): self.store = {}
    def collection(self, _): return self
    def document(self, key): return FakeRef(self.store, key)
    def transaction(self): return object()

# The real claim() uses @firestore.transactional; here we call the same logic
# directly so the SEQUENCE is what you can see, not the plumbing.
def claim(db, doc_key, gcs_uri):
    ref = db.collection("documents").document(doc_key)
    if ref.get().exists:
        return False
    ref.set({"gcs_uri": gcs_uri, "status": "processing"})
    return True

def release(db, doc_key, error):
    db.collection("documents").document(doc_key).set(
        {"status": "failed", "error": error}, merge=True)

db = FakeDB()
KEY = "acme_3f2b9c...";  URI = "gs://documind-uploads/acme/hr_policy_2026.pdf"

print("delivery 1 (first ever):     ", claim(db, KEY, URI))   # True  - process it
print("delivery 2 (Pub/Sub retry):  ", claim(db, KEY, URI))   # False - ack, do nothing
print("re-upload of the same bytes: ", claim(db, KEY, URI))   # False - same key

# Now the failure that used to strand documents for ever: the worker crashed
# between claim and finish, so the claim is still held.
db.store[KEY]["status"] = "processing"      # as it would be after a crash
print("\nafter a crash, retry sees:  ", claim(db, KEY, URI), "<- stuck for ever")
release(db, KEY, "TimeoutError")
del db.store[KEY]                            # release + a reaper clears the claim
print("after release + reaper:      ", claim(db, KEY, URI), "<- retry can proceed")


## Cell 5: Poison
> **400, not 500.** 500 tells Pub/Sub "try again"; a message this worker can never parse fails identically five times, burns the backoff window, and only then reaches the DLQ. 400 acks it at once, and the dead-letter policy still captures it by delivery count.


In [ ]:
# POISON. A zero-byte object is a message the worker can parse - IngestMessage refuses size=0 at the edge - and the
# worker answers 400, not 500: a message that will fail identically every time must not be retried five times with
# backoff before the DLQ sees it. The ingest_poison line is the proof; the dead-letter subscription is where the
# message lands after Pub/Sub's own delivery attempts (eventarc.tf: five, 10 s to 600 s), minutes later.
since = datetime.datetime.now(datetime.timezone.utc)
bad = upload(f"poison_{STAMP}.pdf", b"", "application/pdf")
print("uploaded", bad.split("/")[-1], "(zero bytes)")
line = wait_for("ingest_poison", since, tenant=None, minutes=2)
assert line, "no ingest_poison line in 2 min"
print("the worker refused it (400):", line.get("error"))
print()
print("the dead-letter subscription, peeked without acking (make dlq):")
print(gcloud("pubsub", "subscriptions", "pull", "ingest-dlq-sub", "--limit", "5",
             "--format=table(message.messageId,message.attributes.objectId,message.attributes.eventTime,deliveryAttempt)")
      or "(empty: Pub/Sub is still retrying - eventarc.tf's backoff runs 10 s to 600 s; re-run this cell in a few minutes)")


### The poison path, in shape


In [ ]:
# The poison path, and why it returns 400 rather than 500.
import base64, json

def push(envelope):
    """The worker's entry point, minus GCS and Doc AI."""
    try:
        raw = base64.b64decode(envelope["message"]["data"])
        msg = IngestMessage.model_validate_json(raw)
    except Exception as e:
        return 400, f"unparseable: {type(e).__name__}"
    return 200, f"accepted {msg.gcs_uri}"

good = IngestMessage(bucket=UPLOAD_BUCKET, name=f"{TENANT}/hr_policy_2026.pdf", size=48_000, content_type="application/pdf", generation="17", tenant_id=TENANT)
ok = {"message": {"data": base64.b64encode(good.model_dump_json().encode()).decode()}}
poison = {"message": {"data": base64.b64encode(b'{"bucket": "u"}').decode()}}
notjson = {"message": {"data": base64.b64encode(b'<html>404</html>').decode()}}

for name, env in (("valid", ok), ("missing fields", poison), ("not json", notjson)):
    print(f"  {name:16} -> HTTP {push(env)[0]}  {push(env)[1][:52]}")

# WHY 400 AND NOT 500:
#   500 tells Pub/Sub "try again". A message this worker can never parse will
#   fail identically five times, burn the backoff window, and only THEN reach
#   the DLQ - twenty minutes of retrying something that cannot succeed.
#   400 acks it immediately; the dead_letter_policy still captures it because
#   the message is nacked into the DLQ by delivery count, and the operator sees
#   it in ingest-dlq rather than in a latency graph.
#
# The distinction is: 500 means "this might work later", 400 means "this will
# never work". Retrying the second kind is how a poison message takes a queue
# down.


## Cell 6: A figure is a document


In [ ]:
import io
from PIL import Image, ImageDraw

# A FIGURE IS A DOCUMENT (Module 9). A PNG under the same prefix takes the media path: no Document AI, no chunker -
# Gemini describes the picture, the description becomes ONE chunk of kind figure with media_url set, and the pixels
# are scanned by DLP in the image region before anything is indexed. A citation of it renders inline in the UI (12.4).
img = Image.new("RGB", (640, 320), "white")
d = ImageDraw.Draw(img)
d.rectangle([40, 60, 600, 260], outline="black", width=3)
for i, (label, h) in enumerate((("Year 1", 40), ("Year 3", 90), ("Year 5", 150), ("Year 7", 190))):
    x = 90 + i * 130
    d.rectangle([x, 250 - h, x + 80, 250], fill="#0d9488")
    d.text((x, 255), label, fill="black")
d.text((40, 20), f"Figure: gratuity accrual by years of service (rehearsal {STAMP})", fill="black")
buf = io.BytesIO(); img.save(buf, format="PNG")
since = datetime.datetime.now(datetime.timezone.utc)
png = upload(f"module12_figure_{STAMP}.png", buf.getvalue(), "image/png")
line = wait_for("ingest_ok", since)
assert line, "no ingest_ok for the PNG in 5 min"
print("the worker's line:", {k: line.get(k) for k in ("doc_key", "chunks", "pages", "kinds")})
rows = chunks_of(png)
c = rows[0]
print("the chunk:", {"kind": c.get("kind"), "doc_type": c.get("doc_type"), "media_url": (c.get("media_url") or "")[:60], "caption": c["text"][:120]})
assert c.get("kind") == "figure" and c.get("media_url"), "a PNG becomes one figure chunk with a locator"


## Cell 7: The contracts, exercised
Against the module the lane runs - imported from the clone - so if `contracts.py` is broken, this is where you find out.


In [ ]:
from pydantic import ValidationError

# THE CONTRACTS, EXERCISED - against the module the LANE runs, imported from the clone, not a copy pasted here.
good = IngestMessage(bucket=UPLOAD_BUCKET, name=f"{TENANT}/hr_policy_2026.pdf", size=48_000, content_type="application/pdf",
                     generation="17", tenant_id=TENANT)
print("parsed:", good.gcs_uri)
pdf = b"%PDF-1.7 ... notice period 60 days ..."
a = DocumentContract(tenant_id=TENANT, sha256=sha256_of(pdf), gcs_uri=f"gs://{UPLOAD_BUCKET}/{TENANT}/first.pdf", pages=48)
b = DocumentContract(tenant_id=TENANT, sha256=sha256_of(pdf), gcs_uri=f"gs://{UPLOAD_BUCKET}/{TENANT}/second-upload.pdf", pages=48)
print("same bytes, different object names -> same doc_key:", a.doc_key == b.doc_key, "|", a.doc_key[:34], "...")
print("chunk ids are deterministic and tenant-scoped:", a.chunk_id(0) == b.chunk_id(0), a.chunk_id(0)[:24])
for bad in ({"bucket": "u", "name": "../../etc/passwd", "size": 1, "content_type": "application/pdf", "generation": "1", "tenant_id": TENANT},
            {"bucket": "u", "name": f"{TENANT}/x.pdf", "size": 0, "content_type": "application/pdf", "generation": "1", "tenant_id": TENANT}):
    try:
        IngestMessage(**bad)
        print("ACCEPTED - the validator is not doing its job")
    except ValidationError as e:
        print("rejected:", e.errors()[0]["loc"][0], "-", e.errors()[0]["msg"][:48])


## Cell 8: A policy changes
> **The index has a current.** A document has an identity (its object path) and versions (its content hashes). Re-issue it under the same name and the worker indexes the new version - staged, then swapped current in one pass - reuses every chunk whose text did not change, retires the old version's chunks (a flag and an `expire_at`, never a delete), records the source in `sources/`, and moves the tenant's corpus fingerprint so the API's cache follows the ledger. Upload the old bytes again and they come back without a re-embedding. The date a document declares rides on its chunks, into the context and the stream.


In [ ]:
# A POLICY CHANGES (the ledger, 11 September 2026). The handbook is re-issued under its OWN object name: NP-03's notice
# period becomes 90 days, dated 1 October 2026. The worker indexes revision 2 - staged, then swapped current in one pass -
# reusing every chunk revision 1 already had (the carry-over, by chunk_hash), RETIRES revision 1's chunks - flagged and
# stamped expire_at, never deleted - records the source in sources/, moves the tenant's corpus fingerprint, and says so.
# Then the undo: the original bytes again, and revision 1 comes back without a single re-embedding. The lane ends
# where it began, so the golden row that expects 60 days (lk-06) stays green for the next lesson.
V2 = open(f"{KIT}/deploy/evals/demo/hr_policy_2026_v2.md", "rb").read()
V1 = open(f"{KIT}/deploy/evals/corpus/acme/hr_policy_2026.md", "rb").read()
NAME = "hr_policy_2026.md"                                     # the SAME object name: a document re-issued, not a new one
Q = "What is the notice period for a confirmed E3?"
ledger = db.collection("sources").document(f"{TENANT}~{NAME}")
uri = f"gs://{UPLOAD_BUCKET}/{TENANT}/{NAME}"

def versions() -> dict:
    """current / retired chunk counts for the object, by version - the ledger's promise, read from the index."""
    out = {}
    for d in chunks_of(uri):
        key = d.get("doc_key") or "pre-ledger"
        out.setdefault(key, {"current": 0, "retired": 0})["retired" if d.get("current") is False else "current"] += 1
    return out

def ask() -> str:
    st, ans = api("/v1/query", {"query": Q, "tenant_id": TENANT, "user_id": "u_12_5", "top_k": 5, "stream": False})
    return (ans.get("answer", "") if isinstance(ans, dict) else str(ans))[:160].replace(chr(10), " ")

print("before     :", versions(), "| ledger row:", (ledger.get().to_dict() or {}).get("doc_key", "none yet (a lane older than the ledger: make backfill-current)"))
since = datetime.datetime.now(datetime.timezone.utc)
upload(NAME, V2, "text/markdown")
line = wait_for("ingest_ok", since)
assert line, "no ingest_ok in 5 min"
print("the line   :", {k: line.get(k) for k in ("doc_key", "chunks", "reused", "embedded", "retired", "effective_from", "generation")})
if not line.get("retired"):
    print("the worker on the lane predates the ledger (no retired count). Cloud Shell: make build deploy-services SERVICES_lean=ingest"
          " SCRIPTS_lean=commands/lesson-12.5.sh, make backfill-current, then re-run. Restoring revision 1 now.")
    upload(NAME, V1, "text/markdown")
else:
    row = ledger.get().to_dict()
    print("the ledger :", {k: row.get(k) for k in ("doc_key", "generation", "effective_from", "chunks", "status")})
    print("after      :", versions())
    assert row["doc_key"] == line["doc_key"] and line["effective_from"] == "2026-10-01"
    assert all(v["current"] == 0 for k, v in versions().items() if k != line["doc_key"]), "only revision 2 is current"
    if line.get("reused") is not None:
        assert line["reused"] > line["embedded"], "a one-clause edit reuses most of the handbook (281 of 283 by section)"
        print(f"the cost   : {line['reused']} chunks reused by hash, {line['embedded']} embedded, {line['retired']} retired - pay for what changed")
    time.sleep(10)
    a2 = ask()
    print("revision 2 :", a2)
    if "90" not in a2:
        print("  (both versions were retrieved: the API on the lane predates the ledger - make build deploy-services SERVICES_lean=api)")
    # THE UNDO: the original bytes again. Their chunks are still here, flagged; the worker flips them back
    # (ingest_reactivated) and retires revision 2 - nothing is re-embedded, because nothing was ever deleted.
    since = datetime.datetime.now(datetime.timezone.utc)
    upload(NAME, V1, "text/markdown")
    back = wait_for("ingest_reactivated", since, minutes=2) or wait_for("ingest_ok", since, minutes=3)
    assert back, "no ingest line for the undo in 5 min"
    print("the undo   :", back.get("event"), {k: back.get(k) for k in ("doc_key", "chunks", "reused", "embedded", "retired")}, "| now:", versions())
    time.sleep(10)
    a1 = ask()
    print("revision 1 :", a1)
    assert "60" in a1, "the lane ends where it began: lk-06 stays green"
print("\nmake reindex FILE=evals/demo/hr_policy_2026_v2.md NAME=hr_policy_2026.md TENANT=acme   # the same from Cloud Shell; make reconcile is the nightly half")


### The carry-over, the stale event, the retention
> Three more things the ledger promises: an edit costs what changed, an event that arrives late changes nothing, and a retired row leaves by policy. Each is read back from the lane, never asserted from the page.


In [ ]:
# THE CARRY-OVER, READ OFF THE CLAIMS. Cell 8 re-issued the handbook and undid it; both versions' claims are still here
# (documents/{doc_key}), and revision 2's records what it cost: the chunks whose vectors were REUSED by hash from revision
# 1 and the chunks EMBEDDED - the preamble that gained the effective date, and NP-03. The worker chunks a handbook BY
# SECTION (main.py's _chunk, the same rule as shared/documind_corpus.py) precisely so this number is small: fixed windows
# over the whole file would have moved every boundary and reused nothing. The loader's chunker agrees, offline.
sys.path.insert(0, f"{KIT}/deploy/shared")
from documind_corpus import chunk_document, chunk_hash
v2_key, v1_key = f"{TENANT}_{sha256_of(V2)}", f"{TENANT}_{sha256_of(V1)}"
claim = db.collection("documents").document(v2_key).get().to_dict() or {}
print("revision 2's claim:", {k: claim.get(k) for k in ("status", "chunks", "reused", "embedded", "generation")})
if claim.get("reused") is None:
    print("no reused/embedded on the claim: the worker on the lane predates the carry-over (make build deploy-services SERVICES_lean=ingest)")
else:
    assert claim["reused"] + claim["embedded"] == claim["chunks"] and claim["embedded"] < claim["reused"], claim
    rows = chunks_of(uri)
    v1_hashes = {d.get("chunk_hash") for d in rows if d.get("doc_key") == v1_key}
    changed = [d for d in rows if d.get("doc_key") == v2_key and d.get("chunk_hash") not in v1_hashes]
    print("what was embedded :", sorted((d.get("locator"), d["text"][:70].replace(chr(10), " ")) for d in changed))
    assert {d.get("locator") for d in changed} == {"preamble", "NP-03"}, "the two chunks the edit touched"
def _hashes(raw):
    return {c["chunk_hash"] for c in chunk_document({"text": raw.decode("utf-8"), "slug": "hr_policy_2026", "doc_type": "policy",
                                                     "source_uri": uri}, TENANT)}
h1, h2 = _hashes(V1), _hashes(V2)
print(f"the loader, offline: {len(h2)} chunks in revision 2, {len(h1 & h2)} shared with revision 1, {len(h2 - h1)} new -",
      "the same rule, the same number")


In [ ]:
# A LATE REDELIVERY, BY HAND. Push delivery is at-least-once and not in order: the event for an OLDER generation of this
# object can arrive after the ledger has indexed a newer one, and acting on it would make the old version current again.
# The guard reads the ledger's generation first. Publish the object's own record - the JSON_API_V1 shape Cloud Storage
# sends - with a generation one below the current one, and read the worker's one line: nothing downloaded, nothing changed.
blob = gcs.bucket(UPLOAD_BUCKET).get_blob(f"{TENANT}/{NAME}")
record = {"bucket": UPLOAD_BUCKET, "name": f"{TENANT}/{NAME}", "size": str(blob.size), "contentType": "text/markdown",
          "generation": str(int(blob.generation) - 1)}
since = datetime.datetime.now(datetime.timezone.utc)
r = subprocess.run(["gcloud", "pubsub", "topics", "publish", "documind-ingest", "--project", PROJECT_ID,
                    "--message", json.dumps(record)], capture_output=True, text=True)
assert r.returncode == 0, r.stderr
line = wait_for("ingest_stale_event", since, minutes=2)
assert line, "no ingest_stale_event in 2 min: the worker on the lane predates the generation guard"
print("the line   :", {k: line.get(k) for k in ("name", "generation", "ledger_generation", "reason")})
row = db.collection("sources").document(f"{TENANT}~{NAME}").get().to_dict() or {}
print("the ledger :", {k: row.get(k) for k in ("doc_key", "generation", "status")}, "- untouched; versions:", versions())


In [ ]:
# RETENTION, READ BACK FROM THE POLICY. A retired row is stamped expire_at = superseded_at + RETENTION_DAYS (variables.tf's
# retention_days, 30); the TTL policy on chunks.expire_at (firestore_indexes.tf) is the only thing on the lane that ever
# deletes one, within about a day of the stamp. The policy, then a retired row's stamps, then what the manual twin would do.
ttl = gcloud("firestore", "fields", "ttls", "list", "--collection-group", "chunks", "--format", "json")
policies = json.loads(ttl) if ttl.startswith("[") else []
print("TTL policy :", [(f.get("name", "").rsplit("/", 1)[-1], (f.get("ttlConfig") or {}).get("state")) for f in policies]
      or "none yet: terraform apply with firestore_indexes.tf's chunks_expire_at (the owner's hour)")
retired = [d for d in chunks_of(uri) if d.get("current") is False and d.get("expire_at")]
if retired:
    d = retired[0]
    days = (d["expire_at"] - d["superseded_at"]).days
    print(f"a retired row: superseded {d['superseded_at']:%Y-%m-%d}, expires {d['expire_at']:%Y-%m-%d} = {days} days; superseded_by {d.get('superseded_by', '')[:24]}")
    assert 29 <= days <= 31, "RETENTION_DAYS=30 on the worker"
else:
    print("no retired row carries expire_at yet: the worker on the lane predates the retention stamp")
r = subprocess.run([sys.executable, f"{KIT}/deploy/services/ingest/reconcile.py", "--project", PROJECT_ID, "--purge", "--tenant", TENANT],
                   capture_output=True, text=True, cwd=f"{KIT}/deploy/services/ingest")
print((r.stdout.strip().splitlines() or [r.stderr.strip()[-300:]])[-1])
print("nothing here deletes: --purge prints what the policy will remove on its own; --apply exists for a lane that never applied it")


### Reconciliation, the other half
The worker cannot see a deletion or a lost event; a nightly walk of the bucket against the ledger can. The planner is pure, so it runs here on a made-up bucket before the selftest runs from the clone.


In [ ]:
# RECONCILIATION - THE WALK THE WORKER CANNOT DO. The worker keeps the ledger on the way in, one object at a time; what
# it never sees is an object deleted from the bucket, or one overwritten while its event was lost. reconcile.py is the
# other half every production indexer has (LangChain's full cleanup, Vertex AI Search's FULL reconciliation, Bedrock's
# sync): the bucket's current generations against sources/. The planner is pure - it runs here on a made-up bucket -
# then the module's own selftest runs from the clone, and reconcile.tf's nightly job is read back: 23:30 IST, after
# documind-off has floored the lane, as the worker's own account, and only when RECONCILE_JOB=true.
import reconcile

bucket = [{"name": f"{TENANT}/hr_policy_2026.md", "generation": "9", "tenant_id": TENANT},   # in the ledger, unchanged
          {"name": f"{TENANT}/msa_zeta_2026.md", "generation": "12", "tenant_id": TENANT},  # a newer generation than the ledger saw
          {"name": f"{TENANT}/new_handbook.pdf", "generation": "1", "tenant_id": TENANT}]   # never in the ledger
ledger = {f"{TENANT}~hr_policy_2026.md": {"gcs_uri": f"gs://{UPLOAD_BUCKET}/{TENANT}/hr_policy_2026.md", "doc_key": f"{TENANT}_s1", "generation": "9", "sha256": "s1", "status": "indexed"},
          f"{TENANT}~msa_zeta_2026.md": {"gcs_uri": f"gs://{UPLOAD_BUCKET}/{TENANT}/msa_zeta_2026.md", "doc_key": f"{TENANT}_s2", "generation": "11", "sha256": "s2", "status": "indexed"},
          f"{TENANT}~old_circular.pdf": {"gcs_uri": f"gs://{UPLOAD_BUCKET}/{TENANT}/old_circular.pdf", "doc_key": f"{TENANT}_s3", "generation": "3", "sha256": "s3", "status": "indexed",
                                         "name": f"{TENANT}/old_circular.pdf", "tenant_id": TENANT}}   # gone from the bucket
documents = {f"{TENANT}_s9": {"status": "indexed", "gcs_uri": f"gs://{UPLOAD_BUCKET}/{TENANT}/new_handbook.pdf"}}   # indexed before the ledger existed
for a in reconcile.plan(bucket, ledger, documents):
    print(f"  {a['action']:12} {a['name']:36} {a.get('why', '')}")
print()
# check_bytes is decided once the object is hashed: the same sha under a new generation is a touch (record it, index
# nothing); a sha the per-version claims already know is a backfill (the ledger never heard of it); anything else is
# a re-ingest - the object rewritten onto itself, so the worker's ONE path runs, not a second one.
for sha, row in (("s2", ledger[f"{TENANT}~msa_zeta_2026.md"]), ("s9", None), ("s8", ledger[f"{TENANT}~msa_zeta_2026.md"])):
    print(f"  bytes {sha}: {reconcile.decide_bytes(sha, TENANT, row, documents)}")
print()
# THE DRIFT LINE: the number the night is measured by. A source gone from the bucket, one changed under a lost event and
# one the ledger never saw count one each; a metadata-only change does not. reconcile_done carries it; alerts.tf makes it
# a metric and pages when it stays above zero for two runs.
from collections import Counter
toy_sha = {f"{TENANT}/msa_zeta_2026.md": "s8", f"{TENANT}/new_handbook.pdf": "s9"}
summary = Counter()
for a in reconcile.plan(bucket, ledger, documents):
    act = a["action"]
    if act == "check_bytes":
        act = reconcile.decide_bytes(toy_sha[a["name"]], TENANT, ledger.get(a["name"].replace("/", "~")), documents)
    summary[act] += 1
print(json.dumps({"event": "reconcile_done", **summary, "drift": reconcile.drift_of(summary)}))
print()
r = subprocess.run([sys.executable, f"{KIT}/deploy/services/ingest/reconcile.py", "--selftest"], capture_output=True, text=True)
assert r.returncode == 0, r.stdout + r.stderr
print(r.stdout.strip())
print()
print(excerpt("terraform/reconcile.tf", "resource \"google_cloud_run_v2_job\"", after=8))
print()
print(excerpt("terraform/reconcile.tf", "google_cloud_scheduler_job", after=8))
print()
print("make reconcile TENANT_ONLY=acme prints this plan against the real bucket; APPLY=1 acts. reconcile.tf declares the job on")
print("the ingest image and its 23:30 IST schedule; make reconcile-job is an apply with RECONCILE_JOB=true; the last line of")
print("every run is the drift, and alerts.tf pages when it stays above zero for two nights")


## Where this goes
- **12.6**'s DLP list is the one the worker scanned every chunk and the PNG's pixels with; **12.3** listed it.
- **12.7** ships a new worker image the same way it ships the API: a candidate revision, a gate, a traffic flip.
- **12.2**'s `RETRIEVAL_CURRENT_ONLY=on` is the switch that makes the ledger's promise a pre-filter, and its retriever keeps the newest version per source on its own; `GET /v1/sources` is the ledger through the API; the cache follows the corpus fingerprint. `make reconcile` is the nightly half; `deploy/INDEXING.md` is the whole strategy.
- **The batch lane has its consumer (13 September 2026)**: a document over `MAX_INLINE_PAGES` is queued by the worker and indexed by `documind-ingest-batch` - a Cloud Run job on this image (`batch.tf`, `make batch-job`) running the same `index_document()` with no request deadline, started by the worker as it queues and hourly regardless; `make batch` drains the queue now, `make queued` lists it. **4.6's graph reaches the API**: `make graph TENANT=acme` builds it with the lesson's own `FirestoreGraph` (`shared/documind_graph.py`, verbatim), and `make candidate RETRIEVAL_GRAPH=auto` puts the walk's chunks in front of the dense pool - judged on a candidate first.
- **4.3's and 4.4's stores are mirrors of the ledger (P9, 13 September 2026)**: with `MANAGED_MIRROR=rag_engine|vertex_search|both` the worker copies the version it swaps current - as the text its rows hold - into the tenant's RAG Engine corpus (`make rag-corpus TENANT=`) and / or Vertex AI Search data store (`managed.tf`, `MANAGED_SEARCH=true`), and takes a retired version out; `make managed-status` reads every store against the ledger. Off on the lane, refused unless `RESIDENCY=us`; the `rag_engine` and `vertex_search` backends that read them are P9.4 and P9.5.

## ✅ Lesson 12.5 complete
- ✅ A new document ingested end to end: the worker's ingest_ok line, the claim in Firestore, the fact retrieved through the one retrieve()
- ✅ The same bytes refused by the claim: no second line, the chunk count unchanged
- ✅ A zero-byte object refused at the edge with a 400; the dead-letter subscription peeked
- ✅ A PNG turned into one figure chunk with a locator
- ✅ The contracts, the claim race and the poison path, against the lane's own modules
- ✅ A document re-issued under its own name: revision 2 staged then swapped current, revision 1 retired with an expire_at, the ledger row, the answer moved, the undo without a re-embedding
- ✅ The carry-over read off the claims (281 of 283 chunks reused, the preamble and NP-03 embedded), a late redelivery acked as `ingest_stale_event`, the retention read back from the TTL policy
- ✅ The reconciliation planned offline (retire, re-ingest, backfill, touch), the selftest run from the clone, the nightly job read back from reconcile.tf


## The files this lesson owns
Below are the seventeen heredocs the extractor turns into `deploy/services/ingest/*`, the ingestion Terraform and the worker's deploy: the contracts, the claim and the ledger, the parser, the indexer, the worker, the batch lane's consumer (`batch.py`, 13 September 2026: the queued documents through the worker's own `index_document()`, as a Cloud Run job with no request deadline), the graph builder (`graph.py`: 4.6's extraction, resolution and build over the tenant's current chunks, written with the lesson's `FirestoreGraph` from `shared/documind_graph.py`), the managed mirror (`managed.py`, P9.2: the version the swap made current copied into the tenant's RAG Engine corpus and / or Vertex AI Search data store - 4.3's and 4.4's stores - and out again on retirement, never able to fail an ingest), Vector Search (full profile), the Document AI processor, the Firestore indexes (two vector indexes on `chunks` - the second carries `current` - and the TTL policy on `expire_at`), the Eventarc lane with its floor, the batch job with its invoker grant and hourly schedule (`batch.tf`, `BATCH_JOB=true`), the managed stores (`managed.tf`: a Vertex AI Search data store per tenant with the mirror's schema, `MANAGED_SEARCH=true`), the requirements, the Dockerfile and the deploy. The ledger's edits of 11, 12 and 13 September reached them through `readopt`; the story above uploaded four objects into the lane they describe.


In [ ]:
CONTRACTS_PY = '''
"""Contracts for the ingest lane. Everything crossing a queue is validated."""
import hashlib
import re

from pydantic import BaseModel, ConfigDict, Field, field_validator, model_validator

# The shape of a chunk row (12 September 2026). 1 is the ledger's shape: doc_key, current, indexed_at. 2 adds the
# chunk's own identity (chunk_hash, locator), the embedding stamp (embedding_model, embedding_version) and the
# retention field (expire_at). A backfill is explicit, never guessed from a missing field.
SCHEMA_VERSION = 2


class IngestMessage(BaseModel):
    """The object record Cloud Storage publishes on object.finalized (eventarc.tf's
    notification, payload JSON_API_V1), as we choose to see it.

    Pub/Sub hands you whatever the publisher sent. Parsing it into a model at
    the edge means a malformed message fails HERE, loudly, with a field name -
    instead of three functions later with a KeyError nobody can place.

    The record spells the type `contentType` and its size as a string of digits;
    the alias and pydantic's coercion take both. It carries no tenant: the object
    PATH does - `acme/code_on_wages_2019.pdf` - and the first segment is the
    tenant, which is why evals/upload.sh puts one prefix per tenant and why an
    object at the bucket root is poison, not a default tenant.
    """
    model_config = ConfigDict(populate_by_name=True)

    bucket: str
    name: str
    size: int = Field(ge=1)
    content_type: str = Field(alias="contentType")
    generation: str
    tenant_id: str = ""

    @field_validator("name")
    @classmethod
    def _no_traversal(cls, v: str) -> str:
        if ".." in v or v.startswith("/"):
            raise ValueError("object name escapes its prefix")
        return v

    @model_validator(mode="after")
    def _tenant_from_path(self):
        if not self.tenant_id:
            prefix, sep, rest = self.name.partition("/")
            if not sep or not prefix or not rest:
                raise ValueError("object is not under a tenant prefix")
            self.tenant_id = prefix
        return self

    @property
    def gcs_uri(self) -> str:
        return f"gs://{self.bucket}/{self.name}"


class DocumentContract(BaseModel):
    """What one ingested document looks like once it exists.

    doc_key is the IDEMPOTENCY KEY and it is derived from CONTENT, never from
    the Pub/Sub message id. The message id is stable across REDELIVERY, so it
    would deduplicate a retry - but the same PDF uploaded twice is two
    different messages with two different ids, and that is the duplicate
    users actually create. A content hash catches both.
    """
    tenant_id: str
    sha256: str
    gcs_uri: str
    pages: int
    doc_type: str = "unknown"
    # The ledger (11 September 2026): when the document says it applies from. Declared by the document, never
    # guessed by the pipeline; absent means undated. Carried onto every chunk and every citation.
    effective_from: str | None = None
    schema_version: int = SCHEMA_VERSION

    @property
    def doc_key(self) -> str:
        return f"{self.tenant_id}_{self.sha256}"

    def chunk_id(self, i: int) -> str:
        # Deterministic too: a re-run UPSERTS the same ids instead of adding a
        # second copy of every chunk beside the first.
        #
        # And TENANT-SCOPED, like doc_key. Until 7 Sept 2026 this was f"{sha256}#{i}": two
        # tenants uploading the same Act (the corpus shares seven documents between ACME and
        # Zeta or Globex) wrote the same Firestore documents, and the second ingest overwrote
        # the first tenant's chunks with its own tenant_id. ACME lost every shared document
        # to whichever tenant ingested it last, and the live eval refused eight questions
        # about documents ACME had uploaded. The colon matches the ids the notebooks mint
        # (acme:hr_policy_2026#NP-03); the same id in one tenant's index and another's is a
        # collision, never a saving.
        #
        # The id names the VERSION and the position; it is stable for citations and the golden set. The
        # chunk's identity ACROSS versions is its chunk_hash and its locator (12 September 2026), two
        # fields on the row - a decision recorded in deploy/INDEXING.md.
        return f"{self.tenant_id}:{self.sha256}#{i}"


def sha256_of(data: bytes) -> str:
    return hashlib.sha256(data).hexdigest()


_WS = re.compile(r"\\s+")


def chunk_hash(text: str) -> str:
    """The chunk's identity across versions: the hash of its text with the whitespace collapsed. A re-issued
    document keeps every chunk whose hash it keeps - the carry-over in indexer.py copies their vectors and
    embeds only the rest. Whitespace is collapsed because a re-wrapped paragraph is the same paragraph."""
    return hashlib.sha256(_WS.sub(" ", text).strip().encode("utf-8")).hexdigest()


def is_stale(event_generation, ledger_generation) -> bool:
    """The generation guard. Cloud Storage numbers every version of an object; the ledger records the generation
    it indexed. An event carrying an OLDER generation than the ledger's is a late redelivery (push delivery is
    at-least-once and not in order), and acting on it would make an old version current again. A generation
    the ledger has not seen, or a ledger with none, is never stale."""
    try:
        return int(str(event_generation)) < int(str(ledger_generation))
    except (TypeError, ValueError):
        return False


# A document declares its own effective date: `effective_from: 2026-10-01` (or `Effective from: ...`) in its first
# lines - Markdown front matter or a header line - or `_effective_2026-10-01` in its object name for a PDF nobody
# can edit. Two documents that disagree are then a question of dates, not of which one the reranker liked.
_EFFECTIVE_TEXT = re.compile(r"effective[ _-]?(?:from|date)?\\s*[:=]\\s*(\\d{4}-\\d{2}-\\d{2})", re.I)
_EFFECTIVE_NAME = re.compile(r"_effective_(\\d{4}-\\d{2}-\\d{2})\\.")


def effective_from_of(name: str, text: str | None) -> str | None:
    """The date a document declares, from its object name first, then its first lines; None when undated."""
    m = _EFFECTIVE_NAME.search(name or "")
    if m:
        return m.group(1)
    if text:
        m = _EFFECTIVE_TEXT.search(text[:3000])
        if m:
            return m.group(1)
    return None
'''

with open('contracts.py', 'w') as f: f.write(CONTRACTS_PY)
print('wrote contracts.py')


In [ ]:
IDEMPOTENCY_PY = '''
"""The claim: exactly one worker may process a given document - and the ledger: one current version per document."""
import hashlib
import logging
import os
from datetime import datetime, timezone

from google.cloud import firestore

log = logging.getLogger("documind.ingest")
BATCH = 400          # a Firestore batch holds 500 writes; commit early
# The undo window (12 September 2026). The worker stamps expire_at = now + RETENTION_DAYS on a retired row and the TTL
# policy deletes it after that, so an undo older than this cannot find every row. Read the way the worker reads it -
# variables.tf's retention_days, passed by make deploy-services - so the two never disagree.
RETENTION_DAYS = int(os.environ.get("RETENTION_DAYS", "30"))


def claim(db: firestore.Client, doc_key: str, gcs_uri: str, tenant_id: str, retake: bool = False) -> bool:
    """Claim doc_key. True if THIS caller may proceed, False if someone already did.

    The transaction is the whole point. Read-then-write without one is a race
    with a window measured in milliseconds - and Pub/Sub delivers duplicates
    concurrently, so it is a window that gets hit. Two workers both read
    "absent", both write, and the document is ingested twice.

    The row carries tenant_id (12 September 2026): the MCP server's list_documents filters on the field, never on
    the id's prefix - `acme` must not list `acme_eu_*`. retake=True takes a `superseded` claim back: the worker asks
    for it after reactivate() refused the undo, so the bytes are ingested as a fresh version instead of being acked
    as a duplicate for ever. A `queued` claim belongs to the batch lane and is never retaken.
    """
    ref = db.collection("documents").document(doc_key)
    retakes = ("failed", "superseded") if retake else ("failed",)

    @firestore.transactional
    def _claim(tx: firestore.Transaction) -> bool:
        snap = ref.get(transaction=tx)
        # A claim that ended in `failed` is not a claim, it is a record of one. release()
        # writes it so the error is readable; the NEXT delivery must be allowed to try
        # again, or "give the claim back" gave nothing back: on the first live load every
        # retry of a failed document was acked as a duplicate and the document stayed failed.
        if snap.exists and snap.get("status") not in retakes:
            return False
        tx.set(ref, {"gcs_uri": gcs_uri,
                     "tenant_id": tenant_id,
                     "status": "processing",
                     "claimed_at": firestore.SERVER_TIMESTAMP})
        return True

    won = _claim(db.transaction())
    if not won:
        log.info('{"event":"ingest_duplicate","doc_key":"%s"}', doc_key)
    return won


def finish(db: firestore.Client, doc_key: str, chunks: int, counts: dict | None = None,
           generation: str | None = None) -> None:
    """The claim becomes a record: indexed, how many chunks, how many of their vectors were reused by hash and how
    many embedded (12 September 2026), and the object generation the version came from."""
    row = {"status": "indexed", "chunks": chunks, "indexed_at": firestore.SERVER_TIMESTAMP}
    if counts:
        row["reused"], row["embedded"] = int(counts.get("reused", 0)), int(counts.get("embedded", 0))
    if generation is not None:
        row["generation"] = str(generation)
    db.collection("documents").document(doc_key).set(row, merge=True)


def release(db: firestore.Client, doc_key: str, error: str) -> None:
    """Give the claim back so a retry can take it.

    Without this a crash between claim and finish leaves the document claimed
    for ever, and every redelivery sees "already processing" and does nothing.
    The document is then permanently missing and nothing is alerting, because
    from Pub/Sub's point of view every delivery was acked successfully.
    """
    db.collection("documents").document(doc_key).set(
        {"status": "failed", "error": error[:400],
         "failed_at": firestore.SERVER_TIMESTAMP}, merge=True)


# ------------------------------------------------------------------------------ the ledger (11 September 2026)
# A document has an IDENTITY - its object path - and VERSIONS - the content hashes. `documents/{doc_key}` above is
# the per-version claim; `sources/{source_id}` is the per-document ledger: which version is current, its generation,
# when it was indexed, the date it declares. It is what every production indexer keeps (a record manager keyed on the
# source), and it is what lets a re-issued document RETIRE its predecessor's chunks instead of standing beside them.
# Retiring is a flag, never a delete: the audit story survives, the full profile's mirrors keep their history, and a
# bad re-index is undone by uploading the previous bytes again (reactivate) - nothing is re-embedded.
#
# 12 September 2026 (deploy/INDEXING.md): the guard, the swap, the retention and the fingerprint. An event older than
# the ledger's generation is ignored (stale_generation). A new version lands STAGED and invisible, and one pass flips
# it current and retires the old (swap_versions) - a reader never sees two versions of one source. A retired row is
# stamped expire_at; the Firestore TTL policy in firestore_indexes.tf is the only thing that ever deletes a chunk,
# and nothing here calls delete. Every change refreshes the tenant's corpus fingerprint (ledger/{tenant}); the API's
# cache record carries the fingerprint it was packed from and stops being used when they differ.
#
# The tombstone and the verified undo (12 September 2026, the R05 findings). A source a person retires by hand
# (make retire) is `withdrawn` in sources/ - distinct from `retired` (its object left the bucket) and from `superseded`
# (a re-issue) - and nothing automatic brings it back: reconcile leaves the object where it is, the same bytes again
# are acked, reactivate() refuses; make restore SOURCE= clears it. And reactivate() counts before it flips: the rows
# still here against the claim's chunk count, the retire stamp (retired_at on documents/{doc_key}) against
# RETENTION_DAYS. A shortfall or a closed window is reactivate_incomplete, and the worker re-ingests the bytes instead
# of retiring the newer version over a partial undo.


def source_id_for(tenant_id: str, name: str) -> str:
    """Firestore ids cannot hold '/'; the object path already starts with the tenant prefix."""
    return name.replace("/", "~")


def _doc_key_of(snap) -> str:
    d = snap.to_dict() or {}
    # Chunks written before the ledger carry no doc_key; their id is <tenant>:<sha256>#<i>.
    return d.get("doc_key") or snap.id.split("#")[0].replace(":", "_", 1)


def current_chunks(db: firestore.Client, tenant_id: str, gcs_uri: str, doc_key: str,
                   embedding_model: str | None = None, embedding_version: str | None = None) -> int:
    """How many rows of `doc_key` are current for this source - made with the named embedding, when one is named.

    The other lane's version (13 September 2026). The Module 4 notebooks seed the same collection through
    shared/documind_corpus.py: the same doc_key for the same bytes (a real Act's is its PDF's sha, not its
    mirror's) and the same ledger row - but never the claim, so the worker wins claim() for a version that is
    already here. The handler asks this before it parses anything: a version that is current is done. A row
    without the embedding stamp, or with another embedding, does not count - it could not serve this worker's
    queries - and the ingest proceeds as a fresh version whose carry-over reuses nothing from it."""
    n = 0
    query = (db.collection("chunks").where("tenant_id", "==", tenant_id)
             .where("source_uri", "==", gcs_uri).where("current", "==", True))
    for snap in query.stream():
        if _doc_key_of(snap) != doc_key:
            continue
        d = snap.to_dict() or {}
        if embedding_model and (d.get("embedding_model") != embedding_model
                                or str(d.get("embedding_version")) != str(embedding_version)):
            continue
        n += 1
    return n


def take_batch(db: firestore.Client, doc_key: str) -> bool:
    """The batch job takes a queued claim (13 September 2026): documents/{doc_key} queued -> processing, in a transaction,
    so two runs of the job never index one document twice. False when the claim is not queued any more - indexed,
    processing, failed or gone. The claim keeps its fields (the object, the pages, the generation); ingest_batch/ says
    which run took it."""
    ref = db.collection("documents").document(doc_key)

    @firestore.transactional
    def _take(tx: firestore.Transaction) -> bool:
        snap = ref.get(transaction=tx)
        if not snap.exists or snap.get("status") != "queued":
            return False
        tx.set(ref, {**(snap.to_dict() or {}), "status": "processing", "lane": "batch",
                     "claimed_at": firestore.SERVER_TIMESTAMP})
        return True

    won = _take(db.transaction())
    if won:
        db.collection("ingest_batch").document(doc_key).set(
            {"status": "processing", "taken_at": firestore.SERVER_TIMESTAMP}, merge=True)
    return won


def stale_generation(db: firestore.Client, tenant_id: str, name: str, generation) -> str | None:
    """The generation guard. Returns the ledger's generation when this event's is OLDER than it - a late redelivery
    the worker must ignore - and None when the event is as new as the ledger or newer, or the ledger has no row."""
    snap = db.collection("sources").document(source_id_for(tenant_id, name)).get()
    row = (snap.to_dict() or {}) if snap.exists else {}
    have = row.get("generation")
    try:
        if have and int(str(generation)) < int(str(have)):
            return str(have)
    except (TypeError, ValueError):
        return None
    return None


def _retire(batch_state: list, snap, keep_doc_key, expire_at, effective_to) -> None:
    fields = {"current": False, "superseded_by": keep_doc_key, "superseded_at": firestore.SERVER_TIMESTAMP}
    if expire_at is not None:
        fields["expire_at"] = expire_at            # the TTL policy's field: the platform deletes the row after it
    if effective_to:
        fields["effective_to"] = effective_to      # the successor's effective date closes this version's window
    batch_state[0].update(snap.reference, fields)


def retire_previous(db: firestore.Client, tenant_id: str, gcs_uri: str, keep_doc_key: str | None,
                    chunks_collection: str = "chunks", expire_at=None, effective_to: str | None = None) -> dict:
    """Retire every chunk of `gcs_uri` that is not `keep_doc_key`: current=false, superseded_by, superseded_at, and
    expire_at when a retention is given (the worker passes now + RETENTION_DAYS).

    keep_doc_key=None retires the whole source (reconcile: the object is gone from the bucket). Two equality filters
    need no composite index. Returns the retired keys, ids and count - the worker removes the ids from Vector Search
    on the full profile and logs the count."""
    retired_keys, retired_ids = set(), []
    state, pending = [db.batch()], 0
    query = (db.collection(chunks_collection).where("tenant_id", "==", tenant_id)
             .where("source_uri", "==", gcs_uri))
    for snap in query.stream():
        key = _doc_key_of(snap)
        if (keep_doc_key and key == keep_doc_key) or (snap.to_dict() or {}).get("current") is False:
            continue
        _retire(state, snap, keep_doc_key, expire_at, effective_to)
        retired_keys.add(key)
        retired_ids.append(snap.id)
        pending += 1
        if pending == BATCH:
            state[0].commit()
            state, pending = [db.batch()], 0
    if pending:
        state[0].commit()
    for key in retired_keys:
        db.collection("documents").document(key).set(
            {"status": "superseded", "superseded_by": keep_doc_key,
             "superseded_at": firestore.SERVER_TIMESTAMP,
             "retired_at": firestore.SERVER_TIMESTAMP}, merge=True)     # the undo window's clock (reactivate)
    return {"retired_doc_keys": sorted(retired_keys), "retired_ids": retired_ids,
            "retired_chunks": len(retired_ids)}


def swap_versions(db: firestore.Client, tenant_id: str, gcs_uri: str, new_doc_key: str, expire_at=None,
                  effective_to: str | None = None, chunks_collection: str = "chunks") -> dict:
    """Visibility is a swap, not a stream (12 September 2026). The new version's rows were written staged
    (current=false, staged=true, a one-day expire_at in case nothing ever swaps them); this flips them current -
    and clears the stage marks - then retires every other current row of the source, in that order, in batches of
    400. A document up to ~250 chunks flips in one commit. A longer one flips in two, and the retriever's
    newest-per-source guard (rag-api/retriever.py) is what makes that window invisible: a reader between the
    commits gets the new version only. Returns the activated count with retire_previous's dict."""
    activated, retired_keys, retired_ids = 0, set(), []
    state, pending = [db.batch()], 0
    query = (db.collection(chunks_collection).where("tenant_id", "==", tenant_id)
             .where("source_uri", "==", gcs_uri))
    rows = list(query.stream())
    for snap in rows:                                       # the new version first: a reader never finds no version
        d = snap.to_dict() or {}
        if _doc_key_of(snap) == new_doc_key and d.get("current") is not True:
            state[0].update(snap.reference, {"current": True, "staged": firestore.DELETE_FIELD,
                                             "expire_at": firestore.DELETE_FIELD,
                                             "superseded_by": firestore.DELETE_FIELD,
                                             "superseded_at": firestore.DELETE_FIELD,
                                             "effective_to": firestore.DELETE_FIELD})
            activated += 1
            pending += 1
            if pending == BATCH:
                state[0].commit()
                state, pending = [db.batch()], 0
    for snap in rows:                                       # then the old: a flag, never a delete
        d = snap.to_dict() or {}
        key = _doc_key_of(snap)
        if key == new_doc_key or d.get("current") is False:
            continue
        _retire(state, snap, new_doc_key, expire_at, effective_to)
        retired_keys.add(key)
        retired_ids.append(snap.id)
        pending += 1
        if pending == BATCH:
            state[0].commit()
            state, pending = [db.batch()], 0
    if pending:
        state[0].commit()
    for key in retired_keys:
        db.collection("documents").document(key).set(
            {"status": "superseded", "superseded_by": new_doc_key,
             "superseded_at": firestore.SERVER_TIMESTAMP,
             "retired_at": firestore.SERVER_TIMESTAMP}, merge=True)     # the undo window's clock (reactivate)
    return {"activated": activated, "retired_doc_keys": sorted(retired_keys), "retired_ids": retired_ids,
            "retired_chunks": len(retired_ids)}


def _name_of(gcs_uri: str) -> str:
    """gs://bucket/acme/x.md -> acme/x.md: the object name the ledger keys on (source_id_for)."""
    return gcs_uri.split("/", 3)[3] if gcs_uri.startswith("gs://") and gcs_uri.count("/") >= 3 else gcs_uri


def withdrawn(db: firestore.Client, tenant_id: str, gcs_uri: str) -> bool:
    """The tombstone (12 September 2026): a person retired this source by hand (make retire), and only make restore
    may bring it back. The worker asks before the undo; reactivate() asks again, so no caller can skip it."""
    snap = db.collection("sources").document(source_id_for(tenant_id, _name_of(gcs_uri))).get()
    return bool(snap.exists and (snap.to_dict() or {}).get("status") == "withdrawn")


def reactivate(db: firestore.Client, tenant_id: str, gcs_uri: str, doc_key: str,
               chunks_collection: str = "chunks", retention_days: int | None = None) -> int | None:
    """The undo. The same bytes uploaded again after a newer version retired them: their chunks are still here,
    flagged, so flipping the flag back is a re-index that costs nothing. The caller retires the newer version next.
    The retention stamp goes with the flag: a reactivated row is current, and the TTL must not take it.

    Verified before anything is flipped (12 September 2026). The first undo flipped whatever rows remained and
    the worker retired the newer version on the strength of it - after RETENTION_DAYS the TTL policy had taken the
    rows, zero were flipped, and the source was left with no current version at all. So the rows still here are
    counted against the claim's chunk count, and the retire stamp against RETENTION_DAYS; a shortfall or a closed
    window is `reactivate_incomplete` with both numbers, None comes back, and nothing has changed - the worker
    re-ingests the bytes instead. A withdrawn source is refused the same way (`reactivate_withdrawn`). Otherwise
    the rows flipped, as a count, with the claim's stamps cleared."""
    if withdrawn(db, tenant_id, gcs_uri):
        log.info('{"event":"reactivate_withdrawn","doc_key":"%s","gcs_uri":"%s","hint":"make restore SOURCE="}',
                 doc_key, gcs_uri)
        return None
    query = (db.collection(chunks_collection).where("tenant_id", "==", tenant_id)
             .where("source_uri", "==", gcs_uri))
    rows = [snap for snap in query.stream()
            if _doc_key_of(snap) == doc_key and (snap.to_dict() or {}).get("current") is False]
    claim_snap = db.collection("documents").document(doc_key).get()
    record = (claim_snap.to_dict() or {}) if claim_snap.exists else {}
    expected, stamp = record.get("chunks"), record.get("retired_at")
    window = RETENTION_DAYS if retention_days is None else int(retention_days)
    age = None
    if isinstance(stamp, datetime):                     # a claim from before the stamp has no clock: the count decides
        age = (datetime.now(timezone.utc) - (stamp if stamp.tzinfo else stamp.replace(tzinfo=timezone.utc))).days
    short = expected is not None and len(rows) < int(expected)
    if short or (age is not None and age > window):
        log.warning('{"event":"reactivate_incomplete","doc_key":"%s","rows":%d,"chunks":%s,"age_days":%s,'
                    '"retention_days":%d,"reason":"%s"}', doc_key, len(rows),
                    "null" if expected is None else int(expected), "null" if age is None else age, window,
                    "fewer rows than the claim counted" if short else "retired longer ago than the undo window")
        return None
    n, batch, pending = 0, db.batch(), 0
    for snap in rows:
        batch.update(snap.reference, {"current": True, "superseded_by": firestore.DELETE_FIELD,
                                      "superseded_at": firestore.DELETE_FIELD,
                                      "expire_at": firestore.DELETE_FIELD,
                                      "effective_to": firestore.DELETE_FIELD,
                                      "reactivated_at": firestore.SERVER_TIMESTAMP})
        n += 1
        pending += 1
        if pending == BATCH:
            batch.commit()
            batch, pending = db.batch(), 0
    if pending:
        batch.commit()
    db.collection("documents").document(doc_key).set(
        {"status": "indexed", "superseded_by": firestore.DELETE_FIELD, "retired_at": firestore.DELETE_FIELD,
         "reactivated_at": firestore.SERVER_TIMESTAMP}, merge=True)
    return n


def status_of(db: firestore.Client, doc_key: str) -> str | None:
    snap = db.collection("documents").document(doc_key).get()
    return (snap.to_dict() or {}).get("status") if snap.exists else None


def record_source(db: firestore.Client, tenant_id: str, name: str, gcs_uri: str, doc_key: str,
                  generation: str, sha256: str, chunks: int, effective_from: str | None = None,
                  status: str = "indexed", reused: int = 0, embedded: int = 0, retired: int = 0,
                  embedding_model: str | None = None, embedding_version: str | None = None) -> None:
    """The ledger row: what is current for this object path, since when, and what the last reindex cost - the
    chunks whose vectors were reused by hash, the chunks embedded, the rows retired - and the embedding the
    current version was made with."""
    row = {"tenant_id": tenant_id, "name": name, "gcs_uri": gcs_uri, "doc_key": doc_key,
           "generation": str(generation), "sha256": sha256, "chunks": chunks,
           "effective_from": effective_from, "status": status,
           "reused": int(reused), "embedded": int(embedded), "retired": int(retired),
           "indexed_at": firestore.SERVER_TIMESTAMP}
    if embedding_model:
        row["embedding_model"], row["embedding_version"] = embedding_model, str(embedding_version or "")
    db.collection("sources").document(source_id_for(tenant_id, name)).set(row, merge=True)


def corpus_fingerprint(db: firestore.Client, tenant_id: str) -> tuple[str, int]:
    """The hash of the tenant's sorted current doc_keys: the identity of the corpus a cache was packed from. It changes
    on every reindex, retirement and reactivation, and on nothing else."""
    keys = sorted((s.to_dict() or {}).get("doc_key") or "" for s in
                  db.collection("sources").where("tenant_id", "==", tenant_id).where("status", "==", "indexed").stream())
    return hashlib.sha256("\\n".join(keys).encode("utf-8")).hexdigest()[:16], len(keys)


def refresh_fingerprint(db: firestore.Client, tenant_id: str, event: str) -> str:
    """The cache follows the ledger. Every change to what is current re-computes the tenant's fingerprint into
    ledger/{tenant}; rag-api's cache_manager compares it with the fingerprint on the cache record and runs uncached
    when they differ, until make cache packs the corpus that changed. The record is never deleted here - the API
    reads the mismatch, and says so in its log (cache_stale)."""
    fp, n = corpus_fingerprint(db, tenant_id)
    db.collection("ledger").document(tenant_id).set(
        {"tenant_id": tenant_id, "fingerprint": fp, "versions": n, "last_event": event,
         "updated_at": firestore.SERVER_TIMESTAMP}, merge=True)
    log.info('{"event":"ledger_fingerprint","tenant":"%s","fingerprint":"%s","versions":%d}', tenant_id, fp, n)
    return fp
'''

with open('idempotency.py', 'w') as f: f.write(IDEMPOTENCY_PY)
print('wrote idempotency.py')


In [ ]:
PARSER_PY = '''
"""Doc AI, chosen by residency rather than by preference."""
import io
import os

from google.cloud import documentai
from pypdf import PdfReader, PdfWriter

RESIDENCY = os.environ.get("RESIDENCY", "india")     # india | us

# Layout Parser gives you document STRUCTURE - headings, tables, reading order -
# and it runs in `us` only. Enterprise OCR runs in asia-south1 and gives you
# text plus layout boxes, no semantic structure.
#
# For a DPDP deployment that is not a trade-off you get to make on quality
# grounds: if the document carries personal data of people in India and the
# customer's contract says it stays in India, the processor that runs in `us`
# is not available to you, whatever it would have given you.
PROCESSORS = {
    "india": {"location": "asia-south1", "type": "OCR_PROCESSOR"},
    "us":    {"location": "us",          "type": "LAYOUT_PARSER_PROCESSOR"},
}

# An online (synchronous) request takes at most 15 pages, for OCR, Layout Parser and Form
# Parser alike (docs.cloud.google.com/document-ai/limits). The kit's corpus is thirteen
# Acts averaging 53 pages, so a PDF goes up in 15-page slices - lesson 4.1's slices(), the
# same limit - and its pages come back in order. Batch processing takes 500 pages but is
# asynchronous and needs an output prefix to poll; a dozen online calls inside one push
# request is simpler and stays inside the 600-second ack deadline eventarc.tf sets.
ONLINE_PAGE_LIMIT = 15


def processor_config() -> dict:
    return PROCESSORS[RESIDENCY]


def _client_and_name(project_id: str, processor_id: str):
    cfg = processor_config()
    client = documentai.DocumentProcessorServiceClient(
        client_options={"api_endpoint": f"{cfg['location']}-documentai.googleapis.com"})
    return client, f"projects/{project_id}/locations/{cfg['location']}/processors/{processor_id}"


def _page_texts(doc) -> list[str]:
    """One string per page, so the worker can put a form feed between pages and _chunk()
    can name the page a chunk starts on - the page_start a citation shows.

    OCR fills `pages`, each with a text anchor into `text`. Layout Parser fills
    `document_layout` instead: blocks with their own text and a page span. Either way the
    result is pages in order; a document with neither is one page of whatever text it has."""
    if doc.pages:
        pages = []
        for page in doc.pages:
            parts = [doc.text[int(seg.start_index):int(seg.end_index)]
                     for seg in page.layout.text_anchor.text_segments]
            pages.append("".join(parts))
        return pages
    by_page: dict[int, list[str]] = {}

    def walk(blocks):
        for block in blocks:
            page = int(block.page_span.page_start or 1)
            if block.text_block.text:
                by_page.setdefault(page, []).append(block.text_block.text)
            walk(block.text_block.blocks)
            for row in list(block.table_block.header_rows) + list(block.table_block.body_rows):
                for cell in row.cells:
                    walk(cell.blocks)
            if block.list_block.list_entries:
                for entry in block.list_block.list_entries:
                    walk(entry.blocks)

    walk(doc.document_layout.blocks)
    if by_page:
        last = max(by_page)
        return ["\\n".join(by_page.get(p, [])) for p in range(1, last + 1)]
    return [doc.text]


def _process(client, name: str, content: bytes, mime_type: str) -> list[str]:
    result = client.process_document(
        request=documentai.ProcessRequest(
            name=name,
            raw_document=documentai.RawDocument(content=content, mime_type=mime_type)))
    return _page_texts(result.document)


def parse(project_id: str, processor_id: str, content: bytes,
          mime_type: str) -> tuple[str, int]:
    """Return (text with a form feed between pages, page_count)."""
    client, name = _client_and_name(project_id, processor_id)
    if mime_type == "application/pdf":
        reader = PdfReader(io.BytesIO(content))
        if len(reader.pages) > ONLINE_PAGE_LIMIT:
            pages: list[str] = []
            for first in range(0, len(reader.pages), ONLINE_PAGE_LIMIT):
                writer = PdfWriter()
                for page in reader.pages[first:first + ONLINE_PAGE_LIMIT]:
                    writer.add_page(page)
                buf = io.BytesIO()
                writer.write(buf)
                pages.extend(_process(client, name, buf.getvalue(), mime_type))
            return "\\f".join(pages), len(pages)
    pages = _process(client, name, content, mime_type)
    return "\\f".join(pages), len(pages)
'''

with open('parser.py', 'w') as f: f.write(PARSER_PY)
print('wrote parser.py')


In [ ]:
INDEXER_PY = '''
"""Embed, upsert to Vector Search, mirror into Firestore - and reuse what a re-issued document kept."""
import os

from google.cloud import aiplatform
from google.cloud import firestore
from google.cloud.aiplatform_v1.types import IndexDatapoint
from google.cloud.firestore_v1.vector import Vector
from google import genai

from contracts import SCHEMA_VERSION

EMBED_BATCH = 250          # the regional API's per-request ceiling, in texts
# ... and in tokens: text-embedding-005 takes at most 20,000 tokens per REQUEST, across all
# the texts in it. Forty 500-token chunks is 20,000; every long Act failed on exactly this
# the first time the corpus was loaded. Tokens are estimated at three characters each -
# an overestimate for English, so a batch stops early rather than late.
EMBED_TOKENS = 15_000
CHARS_PER_TOKEN = 3
DRY_RUN = os.environ.get("VECTOR_DRY_RUN") == "1"
# ONE declared embedding, stamped on every row (12 September 2026): variables.tf's embedding_model and
# embedding_version reach this worker and rag-api through the same two variables, so query and document vectors
# come from one model by construction, and a model change is a planned migration (make reembed, deploy/INDEXING.md)
# rather than a silent mismatch. The carry-over below reuses a vector only when its stamp is this one.
EMBEDDING_MODEL = os.environ.get("EMBEDDING_MODEL", "text-embedding-005")
EMBEDDING_VERSION = os.environ.get("EMBEDDING_VERSION", "1")

# Embeddings are REGIONAL. Generation is global-only; this client is neither
# interchangeable with that one nor optional to get right.
_embed = genai.Client(enterprise=True,
                      project=os.environ["GOOGLE_CLOUD_PROJECT"],
                      location="us-central1")


def batches(texts: list[str]) -> list[list[str]]:
    """Batches of at most EMBED_BATCH texts and about EMBED_TOKENS tokens. Send 251 texts, or
    20,001 tokens, and the request fails - not the last item, the whole call - so a
    300-chunk document would index nothing at all."""
    out, cur, cur_tokens = [], [], 0
    for t in texts:
        tokens = max(1, len(t) // CHARS_PER_TOKEN)
        if cur and (len(cur) >= EMBED_BATCH or cur_tokens + tokens > EMBED_TOKENS):
            out.append(cur); cur, cur_tokens = [], 0
        cur.append(t); cur_tokens += tokens
    if cur:
        out.append(cur)
    return out


def embed_all(texts: list[str]) -> list[list[float]]:
    out: list[list[float]] = []
    for batch in batches(texts):
        r = _embed.models.embed_content(
            model=EMBEDDING_MODEL, contents=batch,
            config={"output_dimensionality": 768})
        out.extend([e.values for e in r.embeddings])
    return out


# ------------------------------------------------------------------------- the carry-over (12 September 2026)
def held_vectors(db: firestore.Client, doc, chunks_collection: str = "chunks") -> dict[str, list[float]]:
    """The previous version's CURRENT rows of this source, keyed by chunk_hash - only those made with the embedding
    this worker is configured for. Rows older than schema 2 carry no hash and contribute nothing, which is the
    honest outcome: a vector nobody can prove is the same text is embedded again."""
    out: dict[str, list[float]] = {}
    query = (db.collection(chunks_collection).where("tenant_id", "==", doc.tenant_id)
             .where("source_uri", "==", doc.gcs_uri).where("current", "==", True))
    for snap in query.stream():
        d = snap.to_dict() or {}
        h, vec = d.get("chunk_hash"), d.get("embedding")
        if (h and vec is not None and d.get("embedding_model") == EMBEDDING_MODEL
                and str(d.get("embedding_version")) == str(EMBEDDING_VERSION)):
            out[h] = list(vec)
    return out


def plan_carry_over(chunks: list[dict], held: dict[str, list[float]]) -> tuple[list, list[int]]:
    """Pure: for each chunk, the held vector (by chunk_hash) or None; and the positions that need embedding.
    A one-clause edit of a 283-section handbook comes back as 281 hits and two misses - the clause and the
    preamble that now carries the effective date. tools/check_auth_wiring.py runs this offline."""
    vectors, misses = [], []
    for i, c in enumerate(chunks):
        v = held.get(c.get("chunk_hash") or "")
        vectors.append(v)
        if v is None:
            misses.append(i)
    return vectors, misses


def embed_with_carry_over(db: firestore.Client, doc, chunks: list[dict]) -> tuple[list[list[float]], dict]:
    """Pay for what changed. Returns every chunk's vector, and {reused, embedded}: the counts ingest_ok logs, the
    claim records and the ledger row keeps, so a reindex's cost is a number an operator reads, not a bill they
    discover."""
    held = held_vectors(db, doc)
    vectors, misses = plan_carry_over(chunks, held)
    fresh = embed_all([chunks[i]["text"] for i in misses]) if misses else []
    for i, v in zip(misses, fresh):
        vectors[i] = v
    return vectors, {"reused": len(chunks) - len(misses), "embedded": len(misses)}


def _restricts(tenant_id: str, kind: str, doc_type: str) -> list:
    """The four restrict namespaces every datapoint carries, set HERE, at write time - a filter applied only at
    query time is one forgotten WHERE clause away from a leak. rag-api/retriever.py turns each key of a request's
    `filters` into a Namespace of the same name, so a namespace missing here is a filter that matches nothing.

    tenant_id  what makes one index safe for many customers.
    kind       so a caller can ask for figures only (filters={"kind": "figure"} in rag-api's QueryRequest).
    doc_type   the row's doc_type (12 September 2026): filters={"doc_type": "policy"} used to return an empty
               pool, because the namespace was never written. The value is whatever the row carries - `unknown`
               for a text upload the worker did not classify - the same field the Firestore fallback filters on.
    current    the ledger's (12.5): a new version is current; the worker upserts AFTER the swap and removes the
               retired ids, and the query-time restrict is what a reader asks for."""
    return [IndexDatapoint.Restriction(namespace="tenant_id", allow_list=[tenant_id]),
            IndexDatapoint.Restriction(namespace="kind", allow_list=[kind]),
            IndexDatapoint.Restriction(namespace="doc_type", allow_list=[doc_type]),
            IndexDatapoint.Restriction(namespace="current", allow_list=["true"])]


def to_datapoints(doc, chunks: list[dict],
                  vectors: list[list[float]]) -> list[IndexDatapoint]:
    """chunks are dicts - {text, kind, media_url?, page_start?, start?, end?} - since the
    corpus grew figures and video segments (9.6). Only the text is embedded."""
    return [
        IndexDatapoint(
            datapoint_id=doc.chunk_id(i),
            feature_vector=v,
            restricts=_restricts(doc.tenant_id, c.get("kind", "text"), doc.doc_type),
        )
        for i, (c, v) in enumerate(zip(chunks, vectors))
    ]


def upsert(index_name: str, datapoints: list[IndexDatapoint]) -> None:
    if DRY_RUN:
        print(f"  [dry-run] would upsert {len(datapoints)} datapoints, "
              f"ids {datapoints[0].datapoint_id} .. {datapoints[-1].datapoint_id}")
        return
    # Streaming upserts need an index created with STREAM_UPDATE. A BATCH_UPDATE
    # index accepts the call and applies nothing until the next batch job, which
    # looks exactly like a slow index.
    aiplatform.MatchingEngineIndex(index_name).upsert_datapoints(
        datapoints=datapoints)


def remove_datapoints(index_name: str, ids: list[str]) -> None:
    """The full profile's half of retiring a version: the old ids leave the ANN tier (permanent there - the
    Firestore rows keep the flag and the history, and reupsert() below is how the undo puts them back)."""
    if not ids:
        return
    if DRY_RUN:
        print(f"  [dry-run] would remove {len(ids)} datapoints, ids {ids[0]} .. {ids[-1]}")
        return
    aiplatform.MatchingEngineIndex(index_name).remove_datapoints(datapoint_ids=ids)


def reupsert(index_name: str, db: firestore.Client, tenant_id: str, gcs_uri: str, doc_key: str,
             chunks_collection: str = "chunks") -> int:
    """The undo's half on the full profile (12 September 2026). remove_datapoints() took the retired ids out of the
    ANN tier for good, so an undo that only flipped its Firestore rows left the version current in Firestore and
    absent from Vector Search - retrievable on the lean profile, invisible on the full one. The rows kept their
    vectors (the `embedding` field is the chaos fallback's), so the current rows of the reactivated version go back
    up from there, with the same four restricts to_datapoints() writes, and nothing is embedded. Returns the
    datapoints upserted; the worker calls it BEFORE it retires the newer version, so the tier never holds none."""
    query = (db.collection(chunks_collection).where("tenant_id", "==", tenant_id)
             .where("source_uri", "==", gcs_uri).where("current", "==", True))
    points = []
    for snap in query.stream():
        d = snap.to_dict() or {}
        key = d.get("doc_key") or snap.id.split("#")[0].replace(":", "_", 1)     # idempotency._doc_key_of's rule
        if key != doc_key or d.get("embedding") is None:
            continue
        points.append(IndexDatapoint(datapoint_id=snap.id, feature_vector=list(d["embedding"]),
                                     restricts=_restricts(d.get("tenant_id") or tenant_id, d.get("kind") or "text",
                                                          d.get("doc_type") or "unknown")))
    if points:
        upsert(index_name, points)
    return len(points)


def mirror_to_firestore(db: firestore.Client, doc, chunks: list[dict],
                        vectors: list[list[float]], staged: bool = False, stage_expire_at=None) -> None:
    """The payload store, and the chaos fallback.

    Vector Search holds the vectors; Firestore holds the text the model quotes.
    Storing the embedding here TOO, as a Vector field, means find_nearest() can
    answer while Vector Search is unavailable - slower and good enough, instead
    of an outage.

    The document is THE canonical shape (2.3, 4.2, 4.5, rag-api): tenant_id, text,
    source_uri, page_start, doc_type, embedding - plus, for a figure or a video
    segment, kind / media_url / start / end. resolve() in shared/documind_schemas.py
    reads exactly these names into a Citation, so what is written here is what the
    frontend renders as a thumbnail or a timestamp (gap G7). A text chunk carries
    kind="text" and nothing else new, so nothing written before Module 9 changes.

    Schema 2 (12 September 2026) adds chunk_hash and locator (the chunk's identity across versions),
    the embedding stamp, schema_version - and, with staged=True, a row that is NOT current yet:
    current=false, staged=true, expire_at a day out. idempotency.swap_versions makes it current in one
    pass and clears the stage marks; a stage nothing ever swaps expires by policy, like a retired row.
    """
    batch, pending = db.batch(), 0
    for i, (c, vec) in enumerate(zip(chunks, vectors)):
        ref = db.collection("chunks").document(doc.chunk_id(i))
        row = {"tenant_id": doc.tenant_id, "text": c["text"],
               "source_uri": doc.gcs_uri, "page_start": c.get("page_start"),
               "doc_type": doc.doc_type, "kind": c.get("kind", "text"),
               # The ledger (11 September 2026): which version this chunk belongs to, that it is the
               # current one, and when it landed. The swap flips `current` on the predecessor.
               "doc_key": doc.doc_key, "current": not staged,
               "indexed_at": firestore.SERVER_TIMESTAMP,
               "chunk_hash": c.get("chunk_hash"), "locator": c.get("locator"),
               "embedding_model": EMBEDDING_MODEL, "embedding_version": EMBEDDING_VERSION,
               "schema_version": SCHEMA_VERSION,
               "embedding": Vector(vec)}
        if c.get("section"):
            row["section"] = c["section"]
        if staged:
            row["staged"] = True
            if stage_expire_at is not None:
                row["expire_at"] = stage_expire_at
        if getattr(doc, "effective_from", None):
            row["effective_from"] = doc.effective_from
        for k in ("media_url", "start", "end"):
            if c.get(k) is not None:
                row[k] = c[k]
        batch.set(ref, row)
        pending += 1
        if pending == 400:                     # a Firestore batch holds 500 writes; a long Act is more
            batch.commit()
            batch, pending = db.batch(), 0
    if pending:
        batch.commit()


def mirror_to_bigquery(bq, table: str, doc, chunks: list[dict], pii_chunk_ids: set) -> int:
    """The SQL lane's copy of the REAL chunks (lesson 5.5, gap G9).

    One row per chunk into rag_data.chunk_source - the same canonical names, plus the DLP
    verdict this worker already computed, so 5.5's feature job needs no second scan and no
    BigQuery model to know pii_flag. insertId = chunk id: a retried message re-inserts the
    same rows and BigQuery de-duplicates them. `table` is BQ_CHUNK_TABLE
    (PROJECT.rag_data.chunk_source); unset means the lane is off and nothing is written.
    """
    if not table:
        return 0
    from datetime import datetime, timezone

    now = datetime.now(timezone.utc).isoformat()
    rows = [{"chunk_id": doc.chunk_id(i), "tenant_id": doc.tenant_id, "text": c["text"],
             "source_uri": doc.gcs_uri, "page_start": c.get("page_start"), "page_end": None,
             "doc_type": doc.doc_type, "kind": c.get("kind", "text"), "heading_path": c.get("section"),
             "last_revised_at": None, "pii_flag": doc.chunk_id(i) in pii_chunk_ids,
             "ingested_at": now} for i, c in enumerate(chunks)]
    errors = bq.insert_rows_json(table, rows, row_ids=[r["chunk_id"] for r in rows])
    if errors:
        raise RuntimeError(f"BigQuery rejected {len(errors)} chunk_source row(s): {errors[0]}")
    return len(rows)
'''

with open('indexer.py', 'w') as f: f.write(INDEXER_PY)
print('wrote indexer.py')


In [ ]:
WORKER_PY = '''
"""The Cloud Run worker behind a Pub/Sub push subscription."""
import base64
import io
import json
import logging
import os
import re
import sys
from datetime import datetime, timedelta, timezone

from fastapi import FastAPI, HTTPException, Request
from google import genai
from google.genai import types as gtypes
from google.api_core.exceptions import NotFound
from google.cloud import firestore, storage
from pydantic import BaseModel, ValidationError
from pypdf import PdfReader

# shared/ ships beside the service in the image (see the Dockerfile), the same
# way services/chat consumes documind_tools.
from shared.pii import inspect_image as pii_inspect_image, inspect_many as pii_inspect_many
from shared.audit_log import emit as audit_emit

from contracts import IngestMessage, DocumentContract, chunk_hash, effective_from_of, sha256_of
from idempotency import (claim, current_chunks, finish, reactivate, record_source, refresh_fingerprint, release,
                         retire_previous, source_id_for, stale_generation, status_of, swap_versions, withdrawn)
from indexer import (EMBEDDING_MODEL, EMBEDDING_VERSION, embed_with_carry_over, mirror_to_bigquery,
                     mirror_to_firestore, remove_datapoints, reupsert, to_datapoints, upsert)
from managed import Mirror
from parser import parse

logging.basicConfig(level=logging.INFO, format="%(message)s", stream=sys.stdout)
log = logging.getLogger("documind.ingest")
app = FastAPI()
_db = firestore.Client()
_gcs = storage.Client()
# THE MANAGED MIRROR (P9.2, 13 September 2026): the tenant's RAG Engine corpus and / or Vertex AI Search data store
# (lessons 4.3 and 4.4), kept to the ledger's current versions from here - after the swap, after the undo - and
# never able to fail an ingest (managed.py). MANAGED_MIRROR=off on the lane; anything else is refused at startup
# unless RESIDENCY=us, because neither store keeps the India story.
_mirror = Mirror.from_env(_db)
INDEX_NAME = os.environ.get("VECTOR_INDEX_NAME", "")
PROJECT = os.environ["GOOGLE_CLOUD_PROJECT"]
PROCESSOR_ID = os.environ.get("DOCAI_PROCESSOR_ID", "")    # docai.tf outputs it
REGION = os.environ.get("REGION", "us-central1")
# The batch lane's consumer (13 September 2026): the Cloud Run JOB batch.tf declares on this image, named here once
# make batch-job has declared it. Empty: nothing is started when a document is queued, and the queue waits for
# make batch (gcloud run jobs execute) - the lane says so on the ingest_queued_batch line.
BATCH_JOB = os.environ.get("BATCH_JOB", "")
GEN_MODEL = os.environ.get("GEN_MODEL", "gemini-3.6-flash")
# The SQL lane (5.5, gap G9): PROJECT.rag_data.chunk_source, declared by dataplex.tf. Unset
# means the lane is off; the worker never needs BigQuery to index a document.
BQ_CHUNK_TABLE = os.environ.get("BQ_CHUNK_TABLE", "")
# Retention (12 September 2026): a retired row is stamped expire_at = now + RETENTION_DAYS, and the Firestore TTL
# policy in firestore_indexes.tf deletes it after that - the only deleter on the lane. The number is variables.tf's
# retention_days, passed by make deploy-services; it is the audit window and the undo window at once.
RETENTION_DAYS = int(os.environ.get("RETENTION_DAYS", "30"))
STAGE_HOURS = 24            # a staged version nothing ever swapped expires by the same policy
_bq = None


def _bigquery():
    global _bq
    if _bq is None:
        from google.cloud import bigquery
        _bq = bigquery.Client(project=PROJECT)
    return _bq

# Fail at STARTUP, not on the first document. audit_log.emit refuses to drop an
# event, so an unset AUDIT_BUCKET would surface as a 500 halfway through an
# ingest, release the claim and retry into the DLQ - a configuration mistake
# wearing the costume of a data problem.
if not os.environ.get("AUDIT_BUCKET"):
    raise RuntimeError(
        "AUDIT_BUCKET is not set. The ingest worker records doc.upload and "
        "dlp.finding events; refusing to start without somewhere to put them.")
# parser.py sends a PDF to Document AI in 15-page slices at roughly a second a page, and the
# push subscription allows 600 seconds before it redelivers, so a document this size finishes
# inline with room to spare. Bigger than this goes to the batch lane - a claim document the batch job consumes
# (batch.py, batch.tf: no request deadline, the same pipeline as below, 13 September 2026); the largest file in
# the kit's corpus is under two hundred pages, so the corpus never takes it.
MAX_INLINE_PAGES = 250

# 4.1's chunker, the shape every lesson's corpus has: ~500 tokens per chunk, an
# overlap so a sentence is never cut in half between two chunks.
CHUNK_CHARS, CHUNK_OVERLAP = 2000, 200
# A handbook's sections (12 September 2026): the same two rules shared/documind_corpus.py chunks with, so the
# lane and the notebooks mint the same chunk texts - and a chunk keeps its identity when a paragraph above it
# changes. Fixed windows over a whole document do not: one inserted line at the top moved every window of the
# handbook, and a re-issue that changed one clause reused none of its 92 windows. By section it reuses 281 of 283.
_SECTION = re.compile(r"^## +(.+?) *$", re.M)
_CODE = re.compile(r"^([A-Z][A-Z0-9]{0,7}(?:-[A-Z0-9]{1,6}){1,2})\\b")   # NP-03, IT-SEC-04, MSA-04, GEN-014

# The corpus has four modalities (Module 9) and ONE contract. An uploaded image, video or
# audio file is not parsed for text - it is DESCRIBED, and the description is what gets
# embedded and quoted; the asset rides alongside as media_url (9.6). kind names are
# the shared contract's: figure, table, segment - never a second vocabulary. The key is the
# content type the object.finalized record carries, which `gcloud storage cp`, the UI's
# uploader and 9.4's signed PUT all set from the file - not the extension.
MEDIA_TYPES = {"image/png": "figure", "image/jpeg": "figure",
               "video/mp4": "segment", "audio/mpeg": "segment"}
_gen = None


def _genai() -> genai.Client:
    # Generation is global-only (Gemini 3.x); lazy, so the worker starts without it.
    global _gen
    if _gen is None:
        _gen = genai.Client(enterprise=True, project=PROJECT, location="global")
    return _gen


def _expire_at(days: float) -> datetime:
    return datetime.now(timezone.utc) + timedelta(days=days)


def _pdf_pages(content: bytes) -> int | None:
    """A PDF's page count off its page tree, before any OCR is paid for (12 September 2026). pypdf reads the tree
    and renders nothing - parser.py already opens the file this way to slice it - and the batch decision needs only
    the number. None when the bytes will not parse: Doc AI then gets its turn and says what is wrong with them."""
    try:
        return len(PdfReader(io.BytesIO(content)).pages)
    except Exception:  # noqa: BLE001 - a count is a hint for routing, never a verdict on the document
        return None


def _parse(content: bytes, content_type: str) -> tuple[str, int]:
    """(text, pages). Plain text needs no processor; everything else goes to Doc AI."""
    if content_type.startswith("text/"):
        text = content.decode("utf-8", "replace")
        return text, max(1, text.count("\\f") + 1)
    if not PROCESSOR_ID:
        raise RuntimeError("DOCAI_PROCESSOR_ID is not set (deploy/terraform/docai.tf outputs it)")
    return parse(PROJECT, PROCESSOR_ID, content, content_type)


def _windows(text: str) -> list[tuple[int, str]]:
    """Fixed windows with overlap, ending on a sentence when there is one nearby; (start offset, text)."""
    text = text.strip()
    out, start = [], 0
    while start < len(text):
        end = min(len(text), start + CHUNK_CHARS)
        if end < len(text):
            cut = text.rfind(". ", start + CHUNK_CHARS // 2, end)
            if cut != -1:
                end = cut + 1
        piece = text[start:end].strip()
        if piece:
            out.append((start, piece))
        if end >= len(text):
            break
        start = max(end - CHUNK_OVERLAP, start + 1)
    return out


def _chunk(text: str) -> list[dict]:
    """A document into chunks, each with a LOCATOR that survives an edit above it.

    A handbook - Markdown with `## ` headings - is one chunk per section, the clause code in the heading
    (NP-03, IT-SEC-04) or the section's ordinal as the locator, a long section windowed within itself.
    Anything else is fixed windows. A text upload marks its page breaks with a form feed (the kit's
    real-document mirrors, evals/fetch_real.py, do; so does parser.py between Doc AI pages), and the windows
    are cut PER PAGE (12 September 2026): a window never crosses a page break, a chunk that starts on page 7
    is cited as page 7, and its locator is the page and the window's ordinal on it (p7-1) - the same texts,
    hashes and locators shared/documind_corpus.py mints for the same bytes in a notebook, so a mirror seeded
    there and the same file uploaded here are one set of chunks, not two. Text without form feeds has no
    page to name, and None is more honest than 1. A mirror's provenance header - the leading <!-- ... -->
    fetch_real.py writes - is not content and is dropped the way the loader drops it. Every chunk carries
    the hash of its text: the carry-over in indexer.py matches on it, so a one-clause edit embeds one clause."""
    text = re.sub(r"\\A\\s*<!--.*?-->\\s*", "", text, count=1, flags=re.S)
    heads = list(_SECTION.finditer(text.strip()))
    out = []
    if heads:
        text = text.strip()
        pre = text[:heads[0].start()].strip()
        for k, (_, piece) in enumerate(_windows(pre)):
            out.append({"text": piece, "kind": "text", "page_start": None,
                        "locator": "preamble" + (f"-{k}" if k else ""), "section": None})
        for n, h in enumerate(heads):
            title = h.group(1).strip()
            body = text[h.end(): heads[n + 1].start() if n + 1 < len(heads) else len(text)].strip()
            code = _CODE.match(title)
            key = code.group(1) if code else f"s{n + 1}"
            for k, (_, piece) in enumerate(_windows(f"{title}\\n{body}")):
                out.append({"text": piece, "kind": "text", "page_start": None,
                            "locator": key + (f"-{k}" if k else ""), "section": title})
    else:
        paged = "\\f" in text
        for p, page in enumerate(text.split("\\f"), 1):
            for k, (_, piece) in enumerate(_windows(page)):
                out.append({"text": piece, "kind": "text", "page_start": p if paged else None,
                            "locator": f"p{p}-{k}" if paged else f"w{k}", "section": None})
    for c in out:
        c["chunk_hash"] = chunk_hash(c["text"])
    return out


class Segment(BaseModel):
    start: float
    end: float
    summary: str


def _describe_media(gcs_uri: str, content_type: str) -> list[dict]:
    """9.6: a figure cannot be retrieved as pixels, so the caption IS the retrievable
    body; a video becomes segments with start/end in seconds. from_uri: the asset stays
    in GCS and never passes through this process."""
    kind = MEDIA_TYPES[content_type]
    part = gtypes.Part.from_uri(file_uri=gcs_uri, mime_type=content_type)
    if kind == "figure":
        r = _genai().models.generate_content(
            model=GEN_MODEL,
            contents=[part, "Describe this figure for retrieval: one caption sentence, then the "
                            "key facts it shows, then any table it contains as Markdown."],
            config=gtypes.GenerateContentConfig(
                thinking_config=gtypes.ThinkingConfig(thinking_level="LOW")))
        text = (r.text or "").strip()
        return [{"text": text, "kind": "figure", "media_url": gcs_uri, "locator": "figure",
                 "chunk_hash": chunk_hash(text)}]
    audio = content_type.startswith("audio/")
    what = "recording" if audio else "video"
    shown = "what is said" if audio else "what is said and shown"
    config = dict(response_mime_type="application/json", response_schema=list[Segment],
                  thinking_config=gtypes.ThinkingConfig(thinking_level="LOW"))
    if not audio:
        # LOW: 'what was said and roughly when' does not need to read text off slides (9.4).
        # A resolution is a frame-sampling dial; an audio file has no frames to sample.
        config["media_resolution"] = gtypes.MediaResolution.MEDIA_RESOLUTION_LOW
    # "quoting every number": a summary paraphrases, and the first live town hall came back as "a slight
    # contraction" where the speaker said "fell 5.2 per cent" - the segment was found, the figure was
    # gone. The caption is the quote (9.6): what is not in the segment's text cannot be retrieved by it.
    r = _genai().models.generate_content(
        model=GEN_MODEL,
        contents=[part, f"Split this {what} into segments of at most 60 seconds. For each, give start "
                        f"and end in seconds and a two-sentence summary of {shown}, quoting every number, "
                        f"percentage, amount and name that is spoken exactly as it is said."],
        config=gtypes.GenerateContentConfig(**config))
    return [{"text": s.summary, "kind": "segment", "media_url": gcs_uri,
             "start": s.start, "end": s.end, "locator": f"t{int(s.start)}-{int(s.end)}",
             "chunk_hash": chunk_hash(s.summary)} for s in (r.parsed or [])]


class IngestFailed(Exception):
    """index_document() gave the claim back and logged ingest_failed; the caller decides what a failure is on its
    lane - a 500 on the push path (Pub/Sub retries, then the DLQ), a failed record on the batch path."""


def _run_batch_job() -> str:
    """Start the batch job (batch.tf: documind-ingest-batch) once, through the Cloud Run Jobs API, as this worker's
    own identity - batch.tf grants it run.invoker on the job. A failure here is logged and swallowed by the caller:
    the hourly schedule and make batch drain the same queue, and a document that is queued is not a failed ingest."""
    import google.auth
    from google.auth.transport.requests import AuthorizedSession
    creds, _ = google.auth.default(scopes=["https://www.googleapis.com/auth/cloud-platform"])
    url = f"https://run.googleapis.com/v2/projects/{PROJECT}/locations/{REGION}/jobs/{BATCH_JOB}:run"
    r = AuthorizedSession(creds).post(url, json={}, timeout=30)
    r.raise_for_status()
    return (r.json().get("metadata") or {}).get("name", "")


def _enqueue_batch(doc: DocumentContract, msg: IngestMessage) -> None:
    """The batch lane. A claim document the batch job consumes; no second queue to provision.

    The document WAITS, and its claim says so - `queued`, not `processing` - which is what keeps a redelivery from
    being acked as a duplicate and the nightly reconcile from rewriting the object onto itself: the worker answers
    queued_batch, reconcile's plan reports `queued` for that generation and counts it apart from the drift. The
    record carries everything the consumer needs to fetch the same bytes (the object, its generation, its type),
    and when the job is declared (BATCH_JOB, batch.tf) it is started here, so a queued document is indexed minutes
    later and not at the next hour. The log line says which (13 September 2026)."""
    _db.collection("ingest_batch").document(doc.doc_key).set({
        "tenant_id": doc.tenant_id, "gcs_uri": doc.gcs_uri, "bucket": msg.bucket, "name": msg.name,
        "content_type": msg.content_type, "size": msg.size, "pages": doc.pages, "generation": str(msg.generation),
        "status": "queued", "queued_at": firestore.SERVER_TIMESTAMP})
    _db.collection("documents").document(doc.doc_key).set(
        {"status": "queued", "pages": doc.pages, "generation": str(msg.generation),
         "queued_at": firestore.SERVER_TIMESTAMP}, merge=True)
    consumer = "no job declared (BATCH_JOB unset): make batch runs the queue now, make batch-job declares and schedules it"
    if BATCH_JOB:
        try:
            op = _run_batch_job()
            consumer = f"{BATCH_JOB} started ({op or 'run requested'}); the hourly schedule backstops it"
        except Exception as e:  # noqa: BLE001 - the queue is durable; the schedule and make batch drain it
            consumer = f"{BATCH_JOB} could not be started ({type(e).__name__}); the hourly schedule drains the queue"
            log.warning(json.dumps({"event": "batch_job_start_failed", "tenant": doc.tenant_id, "doc_key": doc.doc_key,
                                    "job": BATCH_JOB, "error": f"{type(e).__name__}: {e}"[:300]}))
    log.warning(json.dumps({"event": "ingest_queued_batch", "tenant": doc.tenant_id, "doc_key": doc.doc_key,
                            "gcs_uri": doc.gcs_uri, "pages": doc.pages, "generation": str(msg.generation),
                            "consumer": consumer}))


@app.post("/")
async def push(request: Request):
    envelope = await request.json()
    try:
        raw = base64.b64decode(envelope["message"]["data"])
        msg = IngestMessage.model_validate_json(raw)
    except (KeyError, ValueError, ValidationError) as e:
        # 400, NOT 500. A message this worker can never parse must not be
        # retried five times before reaching the DLQ - it will fail the same
        # way every time. Ack the poison and let the DLQ hold it.
        log.warning(json.dumps({"event": "ingest_poison", "error": str(e)[:200]}))
        raise HTTPException(400, "unparseable message")

    # THE GENERATION GUARD (12 September 2026). Push delivery is at-least-once and not in order: the event for an
    # older generation of this object can arrive after the ledger has indexed a newer one. Acting on it would make
    # the old version current again. So the ledger's generation is read first, and an older event is acked as
    # stale - one line, nothing downloaded, nothing changed. The bytes are fetched BY GENERATION, never "whatever
    # the object holds now": an event and its bytes are one version. A generation that is gone (overwritten,
    # unversioned bucket) is the same case: the newer generation's own event indexes it.
    older = stale_generation(_db, msg.tenant_id, msg.name, msg.generation)
    if older:
        log.info(json.dumps({"event": "ingest_stale_event", "tenant": msg.tenant_id, "name": msg.name,
                             "generation": msg.generation, "ledger_generation": older, "reason": "older than the ledger"}))
        return {"status": "stale", "generation": msg.generation, "ledger_generation": older}
    blob = _gcs.bucket(msg.bucket).blob(msg.name, generation=int(msg.generation))
    try:
        content = blob.download_as_bytes()
    except NotFound:
        log.info(json.dumps({"event": "ingest_stale_event", "tenant": msg.tenant_id, "name": msg.name,
                             "generation": msg.generation, "reason": "generation gone: the object was overwritten"}))
        return {"status": "stale", "generation": msg.generation}
    doc = DocumentContract(tenant_id=msg.tenant_id, sha256=sha256_of(content),
                           gcs_uri=msg.gcs_uri, pages=0)

    if not claim(_db, doc.doc_key, doc.gcs_uri, doc.tenant_id):
        if status_of(_db, doc.doc_key) == "superseded":
            # THE UNDO (the ledger, 11 September 2026). The same bytes again, after a newer version
            # retired them: the chunks are still here, flagged. Flip them back, retire the newer
            # version in turn, and nothing is re-embedded - because nothing was ever deleted.
            # Unless a person withdrew the source (12 September 2026): the tombstone holds against a
            # redelivery and against the same bytes uploaded again; make restore SOURCE= is the way back.
            if withdrawn(_db, doc.tenant_id, doc.gcs_uri):
                log.info(json.dumps({"event": "ingest_withdrawn", "tenant": doc.tenant_id, "doc_key": doc.doc_key,
                                     "gcs_uri": doc.gcs_uri, "generation": msg.generation,
                                     "hint": "make restore SOURCE= clears the tombstone"}))
                return {"status": "withdrawn", "doc_key": doc.doc_key}
            back = reactivate(_db, doc.tenant_id, doc.gcs_uri, doc.doc_key)
            if back is not None:
                # The full profile first (12 September 2026): the ids remove_datapoints() took out go back up from
                # the rows' own vectors BEFORE the newer version leaves the tier, so a reader there never finds none.
                if INDEX_NAME:
                    reupsert(INDEX_NAME, _db, doc.tenant_id, doc.gcs_uri, doc.doc_key)
                gone = retire_previous(_db, doc.tenant_id, doc.gcs_uri, doc.doc_key, expire_at=_expire_at(RETENTION_DAYS))
                if INDEX_NAME and gone["retired_ids"]:
                    remove_datapoints(INDEX_NAME, gone["retired_ids"])
                if _mirror.active:               # the managed stores follow the undo: the old text back, the newer version out
                    _mirror.after_undo(doc.tenant_id, doc.gcs_uri, doc.doc_key, gone,
                                       {"generation": str(msg.generation), "name": msg.name})
                record_source(_db, doc.tenant_id, msg.name, doc.gcs_uri, doc.doc_key, msg.generation,
                              doc.sha256, back, effective_from_of(msg.name, None),
                              reused=back, embedded=0, retired=gone["retired_chunks"],
                              embedding_model=EMBEDDING_MODEL, embedding_version=EMBEDDING_VERSION)
                fingerprint = refresh_fingerprint(_db, doc.tenant_id, "ingest_reactivated")
                log.info(json.dumps({"event": "ingest_reactivated", "tenant": doc.tenant_id,
                                     "doc_key": doc.doc_key, "chunks": back, "reused": back, "embedded": 0,
                                     "retired": gone["retired_chunks"], "retired_doc_keys": gone["retired_doc_keys"],
                                     "generation": msg.generation, "fingerprint": fingerprint}))
                return {"status": "reactivated", "doc_key": doc.doc_key, "chunks": back}
            # THE UNDO REFUSED (12 September 2026): reactivate_incomplete said why - fewer rows than the claim
            # counted, or a retire stamp past RETENTION_DAYS; the TTL policy has been at the rows. Nothing was
            # flipped and the newer version is still current. The bytes are here, so take the claim back from its
            # superseded record and ingest them as a fresh version below: the carry-over reuses every vector the
            # newer version still holds, and the swap retires it only once the new rows are whole.
            if not claim(_db, doc.doc_key, doc.gcs_uri, doc.tenant_id, retake=True):
                return {"status": "duplicate", "doc_key": doc.doc_key}
        elif status_of(_db, doc.doc_key) == "queued":
            # Handed to the batch lane by an earlier delivery and waiting for the batch job (batch.py) to take it:
            # acked as queued, so the wait is a fact in the log and not a "duplicate" that hides it.
            return {"status": "queued_batch", "doc_key": doc.doc_key}
        else:
            # Already done by an earlier delivery, or by an earlier upload of the
            # same bytes. Returning 200 ACKS the message: this is a success, not a
            # failure, and retrying it would achieve nothing.
            return {"status": "duplicate", "doc_key": doc.doc_key}

    # THE OTHER LANE'S VERSION (13 September 2026). The Module 4 notebooks seed this collection through
    # shared/documind_corpus.py - the same doc_key for the same bytes (a real Act's is its PDF's sha, not its
    # mirror's), the same ledger row - and never the claim, so the claim above is won for a version that is
    # already current. Nothing is parsed or embedded: the claim becomes its record with the rows it holds, the
    # ledger learns this generation (so the nightly walk stops planning a re-ingest), the fingerprint is
    # refreshed, and the line says what was found. The other order needs nothing: seed() skips a version the
    # lane already holds current.
    already = current_chunks(_db, doc.tenant_id, doc.gcs_uri, doc.doc_key, EMBEDDING_MODEL, EMBEDDING_VERSION)
    if already:
        prior = _db.collection("sources").document(source_id_for(doc.tenant_id, msg.name)).get()
        prior = (prior.to_dict() or {}) if prior.exists else {}
        finish(_db, doc.doc_key, already, {"reused": already, "embedded": 0}, msg.generation)
        record_source(_db, doc.tenant_id, msg.name, doc.gcs_uri, doc.doc_key, msg.generation, doc.sha256, already,
                      prior.get("effective_from"), reused=already, embedded=0, retired=0,
                      embedding_model=EMBEDDING_MODEL, embedding_version=EMBEDDING_VERSION)
        fingerprint = refresh_fingerprint(_db, doc.tenant_id, "ingest_already_current")
        log.info(json.dumps({"event": "ingest_already_current", "tenant": doc.tenant_id, "doc_key": doc.doc_key,
                             "gcs_uri": doc.gcs_uri, "generation": msg.generation, "chunks": already,
                             "reused": already, "embedded": 0, "retired": 0, "fingerprint": fingerprint,
                             "seeded_by": prior.get("generation") or "unknown"}))
        return {"status": "already_current", "doc_key": doc.doc_key, "chunks": already}

    try:
        return index_document(doc, msg, content)
    except IngestFailed:
        raise HTTPException(500, "ingest failed")


def index_document(doc: DocumentContract, msg: IngestMessage, content: bytes, lane: str = "push") -> dict:
    """The pipeline, from a claimed document to its record: media described or text parsed, chunked, DLP-scanned,
    the carry-over, the staged write, the swap, the claim, the ledger row, the fingerprint, one ingest_ok line.
    Shared by the two lanes (13 September 2026): the push handler above, inside a request's 600 s, and batch.py,
    a job with no deadline, which passes lane="batch" so the page ceiling that queued the document is not asked
    again. A failure gives the claim back (release), logs ingest_failed and raises IngestFailed; each lane decides
    what that means for it."""
    gone = {"activated": 0, "retired_doc_keys": [], "retired_ids": [], "retired_chunks": 0}
    counts = {"reused": 0, "embedded": 0}
    text = None                                          # a media document has none: the mirror below skips it
    try:
        image_findings = []
        if msg.content_type in MEDIA_TYPES:
            chunks = _describe_media(doc.gcs_uri, msg.content_type)
            pages = 1
            doc = doc.model_copy(update={"pages": 1, "doc_type": MEDIA_TYPES[msg.content_type],
                                        "effective_from": effective_from_of(msg.name, None)})
            # 9.6: the PIXELS are scanned, not only the caption. The caption is Gemini's
            # description of the picture, and a description of an invoice can carry the
            # invoice's PAN in plain text - so the scan below would find it there too, but a
            # picture of a form with a PAN the caption did not mention would sail into the
            # index. Same info-types, same no-quote rule (shared/pii.py); a video is not an
            # image DLP can read and yields nothing here.
            image_findings = pii_inspect_image(content, msg.content_type)
        else:
            # The batch decision BEFORE Doc AI (12 September 2026): a PDF's page count comes off its page tree
            # for nothing, and a document the push lane cannot finish must not pay for OCR it will not use -
            # the first version parsed the whole file and then looked at the count. Anything that is not a
            # PDF is counted by the parser, as before.
            pages = _pdf_pages(content) if msg.content_type == "application/pdf" else None
            if lane == "push" and pages is not None and pages > MAX_INLINE_PAGES:
                doc = doc.model_copy(update={"pages": pages, "effective_from": effective_from_of(msg.name, None)})
                _enqueue_batch(doc, msg)
                return {"status": "queued_batch", "pages": pages}
            text, pages = _parse(content, msg.content_type)
            doc = doc.model_copy(update={"pages": pages,
                                        "effective_from": effective_from_of(msg.name, text)})
            if lane == "push" and pages > MAX_INLINE_PAGES:
                # A 400-page contract will not finish inside a push request's
                # timeout. Hand it to the batch lane and ack.
                _enqueue_batch(doc, msg)
                return {"status": "queued_batch", "pages": pages}
            chunks = _chunk(text)

        # Scan BEFORE indexing. After the upsert the PII is in the index, and
        # "we scanned it afterwards" is a description of a breach, not a control.
        # A DLP failure fails the whole message: it is nacked, retried, and ends
        # in the DLQ where a human decides - because indexing an unscanned
        # document is the exact thing this control exists to prevent.
        # One scan per document, not per chunk: DLP meters requests per minute, and a
        # corpus load from ten workers at a chunk a request was refused (first live load).
        findings = [{**f, "chunk_id": doc.chunk_id(0)} for f in image_findings]
        for i, chunk_findings in enumerate(pii_inspect_many([c["text"] for c in chunks])):
            for f in chunk_findings:
                findings.append({**f, "chunk_id": doc.chunk_id(i)})
        if findings:
            # No quotes, only types and offsets - see shared/pii.py.
            _db.collection("dlp_findings").add({
                "doc_key": doc.doc_key, "tenant_id": doc.tenant_id,
                "findings": findings, "count": len(findings),
                "scanned_at": firestore.SERVER_TIMESTAMP,
            })
            audit_emit("dlp.finding",
                       actor={"tenant_id": doc.tenant_id, "email": "system:ingest"},
                       target={"type": "document", "id": doc.doc_key,
                               "tenant_id": doc.tenant_id},
                       meta={"types": sorted({f["info_type"] for f in findings}),
                             "count": len(findings)})

        # THE CARRY-OVER (12 September 2026): the previous version's current chunks, matched by chunk_hash; their
        # vectors are copied and only the changed chunks are embedded. The counts go on every line below.
        vectors, counts = embed_with_carry_over(_db, doc, chunks)
        # THE SWAP. The new version is written STAGED - current=false, invisible to every reader - and then one
        # pass flips it current and retires the predecessor's rows (a flag, never a delete, expire_at set so the
        # TTL policy purges them after RETENTION_DAYS). A reader between the two steps still finds exactly one
        # version. The full profile's ANN tier follows: the new ids go up after the swap, the retired ids come out.
        mirror_to_firestore(_db, doc, chunks, vectors, staged=True, stage_expire_at=_expire_at(STAGE_HOURS / 24))
        gone = swap_versions(_db, doc.tenant_id, doc.gcs_uri, doc.doc_key,
                             expire_at=_expire_at(RETENTION_DAYS), effective_to=doc.effective_from)
        if INDEX_NAME:                       # the full profile; the lean one has no index
            upsert(INDEX_NAME, to_datapoints(doc, chunks, vectors))
            if gone["retired_ids"]:
                remove_datapoints(INDEX_NAME, gone["retired_ids"])
        # THE MANAGED MIRROR (P9.2): the version that just became current goes to the tenant's managed stores as the
        # text these rows hold, and the versions the swap retired leave them - one line per store, never a failed
        # ingest (managed.py). A media version stays on this index alone (the plan's D4).
        if _mirror.active and text is not None:
            _mirror.after_swap(doc, text, gone, {"generation": str(msg.generation), "name": msg.name})
        # The SQL lane reads the REAL chunks (5.5, gap G9): the same rows, with the verdict
        # the scan above just produced, so pii_flag in BigQuery is this worker's - never a
        # second scanner's that could disagree.
        mirror_to_bigquery(_bigquery() if BQ_CHUNK_TABLE else None, BQ_CHUNK_TABLE, doc, chunks,
                           {f["chunk_id"] for f in findings})
        finish(_db, doc.doc_key, len(chunks), counts, msg.generation)
        record_source(_db, doc.tenant_id, msg.name, doc.gcs_uri, doc.doc_key, msg.generation,
                      doc.sha256, len(chunks), doc.effective_from,
                      reused=counts["reused"], embedded=counts["embedded"], retired=gone["retired_chunks"],
                      embedding_model=EMBEDDING_MODEL, embedding_version=EMBEDDING_VERSION)
        # The cache follows the ledger: the tenant's corpus fingerprint changes, the API sees its cache record no
        # longer matches and answers uncached, make cache rebuilds the pack from the corpus that changed (12.6, 10.2).
        fingerprint = refresh_fingerprint(_db, doc.tenant_id, "ingest_ok")
        if gone["retired_chunks"]:
            log.info(json.dumps({"event": "ingest_superseded", "tenant": doc.tenant_id,
                                 "doc_key": doc.doc_key, "gcs_uri": doc.gcs_uri,
                                 "retired_doc_keys": gone["retired_doc_keys"],
                                 "retired_chunks": gone["retired_chunks"],
                                 "expire_days": RETENTION_DAYS, "effective_to": doc.effective_from}))

        # The document is now retrievable. Record that, with who and what - the
        # upload event the audit trail is missing without it.
        audit_emit("doc.upload",
                   actor={"tenant_id": doc.tenant_id, "email": "system:ingest"},
                   target={"type": "document", "id": doc.doc_key,
                           "tenant_id": doc.tenant_id},
                   meta={"gcs_uri": doc.gcs_uri, "pages": doc.pages,
                         "chunks": len(chunks), "pii": bool(findings),
                         "kinds": sorted({c["kind"] for c in chunks}),
                         "reused": counts["reused"], "embedded": counts["embedded"],
                         "retired": gone["retired_chunks"]})
    except Exception as e:
        # Give the claim back before failing, or the retry finds the document
        # already claimed and does nothing - for ever. And SAY what failed, on the log
        # line an operator reads first: the claim document carries the same text, but
        # the first live load was diagnosed from request logs that only said 500.
        release(_db, doc.doc_key, f"{type(e).__name__}: {e}")
        log.error(json.dumps({"event": "ingest_failed", "tenant": doc.tenant_id, "lane": lane,
                              "doc_key": doc.doc_key, "gcs_uri": doc.gcs_uri,
                              "error": f"{type(e).__name__}: {e}"[:600]}))
        raise IngestFailed(f"{type(e).__name__}: {e}") from e

    log.info(json.dumps({"event": "ingest_ok", "tenant": doc.tenant_id, "lane": lane,
                         "doc_key": doc.doc_key, "chunks": len(chunks),
                         "pages": pages, "kinds": sorted({c["kind"] for c in chunks}),
                         "reused": counts["reused"], "embedded": counts["embedded"],
                         "retired": gone["retired_chunks"], "generation": msg.generation,
                         "effective_from": doc.effective_from, "fingerprint": fingerprint}))
    return {"status": "indexed", "chunks": len(chunks), "reused": counts["reused"], "embedded": counts["embedded"]}
'''

with open('main.py', 'w') as f: f.write(WORKER_PY)
print('wrote main.py')


In [ ]:
BATCH_PY = '''
#!/usr/bin/env python3
"""The batch lane's consumer (13 September 2026): the documents the push path handed off.

    python batch.py --project P                 # drain ingest_batch/: every queued claim, oldest first
    python batch.py --project P --queued        # print the queue and exit (make queued)
    python batch.py --project P --limit 1 --dry-run

A document over MAX_INLINE_PAGES cannot finish inside a push request's 600 seconds, so the worker writes a claim
(`ingest_batch/{doc_key}`, `documents/{doc_key}` = queued) and returns. This is the consumer: a Cloud Run JOB on the
same image (batch.tf, `make batch-job`), with no request deadline, that the worker starts when it queues a document
(BATCH_JOB) and a schedule starts hourly regardless (`make batch` starts it by hand). It takes each queued claim in a
transaction (two runs never index one document twice), downloads the bytes BY GENERATION, and runs the worker's own
pipeline - main.index_document(): media or text, chunks, DLP, the carry-over, the staged write, the swap, the claim,
the ledger row, the fingerprint - with lane="batch", so the page ceiling that queued the document is not consulted
again. A failure leaves the claim `failed` with the error (release) and the batch record says the same; the next run
does not retake it - make reindex, or the same bytes uploaded again, is the way back. A generation that is gone from
the bucket (overwritten, unversioned) is recorded as gone: the newer generation's own event indexes what is there now.
"""
from __future__ import annotations

import argparse
import json
import logging
import os
import sys

log = logging.getLogger("documind.ingest")


def queued(db, limit: int | None = None) -> list[dict]:
    """The queue, oldest first: every ingest_batch/ record whose status is queued."""
    rows = []
    for snap in db.collection("ingest_batch").where("status", "==", "queued").stream():
        d = snap.to_dict() or {}
        d["doc_key"] = snap.id
        rows.append(d)
    rows.sort(key=lambda r: (str(r.get("queued_at") or ""), r["doc_key"]))
    return rows[:limit] if limit else rows


def _split(gcs_uri: str) -> tuple[str, str]:
    bucket, _, name = gcs_uri[len("gs://"):].partition("/")
    return bucket, name


def run_one(db, gcs, rec: dict, worker) -> str:
    """One queued claim through the worker's pipeline. Returns indexed | skipped | gone | failed."""
    from google.api_core.exceptions import NotFound
    from contracts import DocumentContract, IngestMessage, sha256_of
    from idempotency import release, take_batch

    doc_key = rec["doc_key"]
    if not take_batch(db, doc_key):
        log.info(json.dumps({"event": "batch_skipped", "doc_key": doc_key, "reason": "the claim is not queued any more"}))
        return "skipped"
    bucket, name = (rec.get("bucket"), rec.get("name"))
    if not bucket or not name:
        bucket, name = _split(rec["gcs_uri"])
    record = db.collection("ingest_batch").document(doc_key)
    try:
        content = gcs.bucket(bucket).blob(name, generation=int(rec["generation"])).download_as_bytes()
    except NotFound:
        release(db, doc_key, "generation gone: the object was overwritten before the batch lane read it")
        record.set({"status": "gone"}, merge=True)
        log.info(json.dumps({"event": "batch_gone", "doc_key": doc_key, "gcs_uri": rec["gcs_uri"],
                             "generation": rec["generation"]}))
        return "gone"
    msg = IngestMessage(bucket=bucket, name=name, size=max(1, int(rec.get("size") or len(content))),
                        contentType=rec.get("content_type") or "application/pdf",
                        generation=str(rec["generation"]), tenant_id=rec["tenant_id"])
    doc = DocumentContract(tenant_id=rec["tenant_id"], sha256=sha256_of(content), gcs_uri=rec["gcs_uri"],
                           pages=int(rec.get("pages") or 0))
    if doc.doc_key != doc_key:
        release(db, doc_key, f"the bytes at generation {rec['generation']} hash to {doc.doc_key}, not this claim")
        record.set({"status": "failed", "error": "content hash does not match the claim"}, merge=True)
        return "failed"
    try:
        out = worker.index_document(doc, msg, content, lane="batch")
    except worker.IngestFailed as e:                   # the claim is released and ingest_failed logged already
        record.set({"status": "failed", "error": str(e)[:300]}, merge=True)
        return "failed"
    record.set({"status": "indexed", "chunks": int(out.get("chunks", 0)),
                "reused": int(out.get("reused", 0)), "embedded": int(out.get("embedded", 0))}, merge=True)
    return "indexed"


def main() -> int:
    ap = argparse.ArgumentParser(description=__doc__.splitlines()[0])
    ap.add_argument("--project", default=os.environ.get("GOOGLE_CLOUD_PROJECT"))
    ap.add_argument("--limit", type=int, help="at most this many documents this run")
    ap.add_argument("--queued", action="store_true", help="print the queue and exit")
    ap.add_argument("--dry-run", action="store_true", help="say what would run; take and index nothing")
    args = ap.parse_args()
    logging.basicConfig(level=logging.INFO, format="%(message)s", stream=sys.stdout)
    from google.cloud import firestore
    db = firestore.Client(project=args.project)
    q = queued(db, args.limit)
    if args.queued or args.dry_run:
        print(f"{len(q)} queued document(s)" + (" (limit applied)" if args.limit else ""))
        for r in q:
            print(f"  {r['doc_key'][:20]}  {r.get('gcs_uri', '')}  pages={r.get('pages')}  generation={r.get('generation')}")
        return 0
    if not q:
        log.info(json.dumps({"event": "batch_run", "queued": 0, "indexed": 0, "failed": 0}))
        return 0
    from google.cloud import storage
    import main as worker                                # the worker's module: its clients, its pipeline, its env
    gcs = storage.Client(project=args.project)
    tally = {"indexed": 0, "skipped": 0, "gone": 0, "failed": 0}
    for rec in q:
        tally[run_one(db, gcs, rec, worker)] += 1
    log.info(json.dumps({"event": "batch_run", "queued": len(q), **tally}))
    return 1 if tally["failed"] else 0


if __name__ == "__main__":
    sys.exit(main())
'''

with open('batch.py', 'w') as f: f.write(BATCH_PY)
print('wrote batch.py')


In [ ]:
GRAPH_PY = '''
#!/usr/bin/env python3
"""Build a tenant's knowledge graph from its current chunks - lesson 4.6, on the lane (13 September 2026).

    python graph.py --project P --tenant acme [--limit 200] [--rebuild] [--dry-run] [--ask "Which Acts ..."]

4.6's pipeline, as a tool: every current text chunk of the tenant (the handbook's GEN- boilerplate skipped) through
gemini-3.1-flash-lite with the lesson's GraphExtraction schema and system prompt - entities and relations STATED in
the passage, never inferred; surface forms resolved to one canonical name by the lesson's two passes (normalise, then
text-embedding-005 cosine at 0.92 - a wrong merge is worse than a duplicate node); nodes and edges built the lesson's
way (a stable id from the canonical name, the chunk ids that make citations possible, the best confidence per edge);
written through shared/documind_graph.FirestoreGraph - the same class the notebook runs - into graph_nodes and
graph_edges beside the chunks. An extraction is cached in graph_extractions/{tenant}:{chunk_id} under the chunk's
hash, so a rerun pays for new or changed chunks only. --rebuild erases the tenant's graph first; --limit is the bill
(200 chunks is a few rupees; ACME's 1,600 are priced in 4.6's Cell 12). --ask walks the graph for one question and
prints the seeds, the nodes and the chunk ids the API would put in front of its dense pool (RETRIEVAL_GRAPH=on|auto).
Needs, on Cloud Shell: pip install --user google-genai==2.22.0 google-cloud-firestore==2.30.0 numpy.
"""
from __future__ import annotations

import argparse
import hashlib
import json
import logging
import os
import re
import sys
import time

from pydantic import BaseModel, Field
from typing import List, Literal

log = logging.getLogger("documind.ingest")

EXTRACT_MODEL = "gemini-3.1-flash-lite"   # bulk extraction: cheapest per token (4.6)
EMBED_MODEL = os.environ.get("EMBEDDING_MODEL", "text-embedding-005")
RESOLVE_THRESHOLD = 0.92                  # deliberately high: a wrong merge is worse than a duplicate node (4.6)


class Entity(BaseModel):
    name: str = Field(description="Surface form exactly as written in the text")
    type: Literal["person", "org", "product", "policy", "system", "location", "date"]


class Relation(BaseModel):
    source: str = Field(description="name of the source entity, exactly as in entities")
    target: str = Field(description="name of the target entity, exactly as in entities")
    rel: str = Field(description="UPPER_SNAKE verb phrase, e.g. OWNS, REPORTS_TO, SUPERSEDES")
    confidence: float = Field(ge=0, le=1)


class GraphExtraction(BaseModel):
    entities: List[Entity]
    relations: List[Relation]


EXTRACT_SYSTEM = """You build a knowledge graph from enterprise documents.
Extract only entities and relations STATED in the passage. Never infer, never add
world knowledge. Every relation's source and target must appear in entities.
If the passage states no relation, return an empty relations list."""


def normalise(name: str) -> str:
    """Cheap first pass: case, punctuation and the corporate suffixes that create duplicates (4.6)."""
    n = name.lower().strip()
    n = re.sub(r"[\\.,]", "", n)
    n = re.sub(r"\\b(private|pvt|limited|ltd|inc|llc|corp|corporation|co)\\b", "", n)
    return re.sub(r"\\s+", " ", n).strip()


def node_id(canonical_name: str) -> str:
    """Stable id from the canonical name, so re-ingesting a document updates rather than duplicates (4.6)."""
    return hashlib.sha1(canonical_name.encode("utf-8")).hexdigest()[:32]


def build_graph(extractions: list, canon_of: dict) -> tuple:
    """Nodes and edges from the extractions - pure Python, no store (4.6's build_graph)."""
    nodes, edges = {}, {}
    for x in extractions:
        cid, g = x["chunk_id"], x["graph"]
        for e in g.entities:
            canonical = canon_of.get(e.name, e.name)
            nid = node_id(canonical)
            n = nodes.setdefault(nid, {"name": canonical, "kind": e.type, "chunks": set()})
            n["chunks"].add(cid)                     # this is what makes citations possible
        for r in g.relations:
            src, dst = canon_of.get(r.source), canon_of.get(r.target)
            if not src or not dst or src == dst:     # drop dangling and self edges
                continue
            key = (node_id(src), node_id(dst), r.rel)
            prev = edges.get(key)
            if prev is None or r.confidence > prev["confidence"]:
                edges[key] = {"chunk_id": cid, "confidence": float(r.confidence)}
    return nodes, edges


def resolve_entities(names: list, embed) -> dict:
    """Map every surface form to one canonical name: exact match after normalise(), then cosine over embeddings
    for the survivors (4.6's resolve_entities). `embed(names) -> unit vectors` is the caller's."""
    import numpy as np
    canon_of, buckets = {}, {}
    for n in names:
        buckets.setdefault(normalise(n), []).append(n)
    reps = sorted(buckets)
    if not reps:
        return canon_of
    vecs = np.array(embed([buckets[k][0] for k in reps]), dtype=np.float32)
    vecs = vecs / np.linalg.norm(vecs, axis=1, keepdims=True)
    merged_into = {}
    for i in range(len(reps)):
        if reps[i] in merged_into:
            continue
        for j in range(i + 1, len(reps)):
            if reps[j] in merged_into:
                continue
            if float(vecs[i] @ vecs[j]) >= RESOLVE_THRESHOLD:
                merged_into[reps[j]] = reps[i]
    for key, surfaces in buckets.items():
        target = merged_into.get(key, key)
        canonical = buckets[target][0]
        for s in surfaces:
            canon_of[s] = canonical
    return canon_of


def load_chunks(db, tenant: str, limit: int | None) -> list[dict]:
    """The tenant's current text chunks, in reading order per source; GEN- boilerplate and media skipped."""
    rows = []
    for snap in db.collection("chunks").where("tenant_id", "==", tenant).stream():
        x = snap.to_dict() or {}
        if x.get("current") is False or x.get("kind", "text") != "text" or (x.get("section") or "").startswith("GEN-"):
            continue
        rows.append({"chunk_id": snap.id, "text": x.get("text", ""), "source_uri": x.get("source_uri", ""),
                     "chunk_hash": x.get("chunk_hash") or hashlib.sha256(re.sub(r"\\s+", " ", x.get("text", "")).strip().encode()).hexdigest(),
                     "locator": x.get("locator") or ""})
    rows.sort(key=lambda r: (r["source_uri"], r["locator"], r["chunk_id"]))
    return rows[:limit] if limit else rows


def extract_all(db, tenant: str, chunks: list[dict], gen, pause: float = 0.1) -> list[dict]:
    """One typed subgraph per chunk, cached under the chunk's hash; one bad chunk never stops the build."""
    from google.genai import types
    out, fresh = [], 0
    for i, c in enumerate(chunks, 1):
        ref = db.collection("graph_extractions").document(f"{tenant}:{c['chunk_id']}".replace("/", "~"))
        snap = ref.get()
        cached = (snap.to_dict() or {}) if snap.exists else {}
        if cached.get("chunk_hash") == c["chunk_hash"]:
            out.append({"chunk_id": c["chunk_id"], "graph": GraphExtraction.model_validate(cached["graph"])})
            continue
        try:
            r = gen.models.generate_content(
                model=EXTRACT_MODEL, contents=f"Passage:\\n{c['text']}",
                config=types.GenerateContentConfig(
                    system_instruction=EXTRACT_SYSTEM, response_mime_type="application/json",
                    response_schema=GraphExtraction,
                    thinking_config=types.ThinkingConfig(thinking_level="LOW")))
            g = r.parsed if isinstance(r.parsed, GraphExtraction) else GraphExtraction.model_validate_json(r.text or "{}")
        except Exception as e:  # noqa: BLE001 - one bad chunk must not stop the ingest
            log.warning(json.dumps({"event": "graph_extract_failed", "tenant": tenant, "chunk_id": c["chunk_id"],
                                    "error": f"{type(e).__name__}: {e}"[:200]}))
            continue
        ref.set({"tenant_id": tenant, "chunk_id": c["chunk_id"], "chunk_hash": c["chunk_hash"],
                 "graph": g.model_dump(), "model": EXTRACT_MODEL})
        out.append({"chunk_id": c["chunk_id"], "graph": g})
        fresh += 1
        if i % 25 == 0:
            log.info(json.dumps({"event": "graph_extract_progress", "tenant": tenant, "done": i, "of": len(chunks), "fresh": fresh}))
        time.sleep(pause)                            # stay well inside the per-minute quota
    log.info(json.dumps({"event": "graph_extracted", "tenant": tenant, "chunks": len(chunks), "extracted": len(out), "fresh": fresh}))
    return out


def main() -> int:
    ap = argparse.ArgumentParser(description=__doc__.splitlines()[0])
    ap.add_argument("--project", default=os.environ.get("GOOGLE_CLOUD_PROJECT"))
    ap.add_argument("--tenant", default="acme")
    ap.add_argument("--limit", type=int, help="chunks to extract, in reading order (the bill)")
    ap.add_argument("--rebuild", action="store_true", help="erase the tenant's graph first")
    ap.add_argument("--dry-run", action="store_true", help="count the chunks; extract nothing")
    ap.add_argument("--ask", help="walk the graph for one question and print what the API would fetch")
    ap.add_argument("--hops", type=int, default=1)
    args = ap.parse_args()
    logging.basicConfig(level=logging.INFO, format="%(message)s", stream=sys.stdout)
    from google.cloud import firestore
    from shared.documind_graph import FirestoreGraph, graph_chunk_ids
    db = firestore.Client(project=args.project)
    if args.ask:
        r = graph_chunk_ids(db, args.ask, args.tenant, hops=args.hops)
        print(json.dumps({"question": args.ask, "seeds": [s["name"] for s in r["seeds"]],
                          "nodes": [n["name"] for n in r["nodes"]], "chunk_ids": r["chunk_ids"]}, indent=1))
        return 0
    chunks = load_chunks(db, args.tenant, args.limit)
    print(f"{len(chunks)} current text chunks for tenant {args.tenant!r}" + (f" (first {args.limit})" if args.limit else ""))
    if args.dry_run:
        return 0
    from google import genai
    from google.genai import types
    gen = genai.Client(enterprise=True, project=args.project, location="global")       # generation: global only
    emb = genai.Client(enterprise=True, project=args.project, location=os.environ.get("EMBED_LOCATION", "us-central1"))

    def embed(names: list) -> list:
        vecs = []
        for i in range(0, len(names), 250):
            r = emb.models.embed_content(model=EMBED_MODEL, contents=names[i:i + 250],
                                         config=types.EmbedContentConfig(task_type="SEMANTIC_SIMILARITY", output_dimensionality=768))
            vecs.extend(e.values for e in r.embeddings)
        return vecs

    extractions = extract_all(db, args.tenant, chunks, gen)
    canon_of = resolve_entities([e.name for x in extractions for e in x["graph"].entities], embed)
    nodes, edges = build_graph(extractions, canon_of)
    graph = FirestoreGraph(db)
    if args.rebuild:
        graph.delete_tenant(args.tenant)
    graph.load(nodes, edges, args.tenant)
    log.info(json.dumps({"event": "graph_built", "tenant": args.tenant, "chunks": len(chunks), "surface_forms": len(canon_of),
                         "nodes": len(nodes), "edges": len(edges)}))
    return 0


if __name__ == "__main__":
    sys.exit(main())
'''

with open('graph.py', 'w') as f: f.write(GRAPH_PY)
print('wrote graph.py')


In [ ]:
MANAGED_PY = '''
#!/usr/bin/env python3
"""The managed mirror (P9.2, 13 September 2026): the ledger's current versions, copied into the tenant's Vertex AI
RAG Engine corpus (lesson 4.3) and / or Vertex AI Search data store (lesson 4.4) - the stores the rag_engine and
vertex_search retrieval backends read (P9.4, P9.5).

    python managed.py --project P --status [--tenant acme]          # each tenant, each store, held against the ledger (make managed-status)
    python managed.py --project P --create-corpus --tenant acme     # the tenant's RAG Engine corpus, once, by name (make rag-corpus)

The rule (managed-retrieval-plan-2026-09-13.md, D3): the ledger is the source of truth and a store holds current
versions only. The worker calls after_swap() once swap_versions() has made a version current - with the text it
indexed, so the store carries what the kit's rows carry and a managed context can be matched back to a row - and
after_undo() when the undo made an older version current again (its text read off its own rows: no second parse, no
Document AI call); retired() when versions leave (a retirement by the walk, a withdrawal by hand). The managed
document id is the doc_key on both stores - a Vertex AI Search Document.id, a RagFile's display name - so every call
is idempotent and a listing compares with the ledger. A tenant with no store is mirror_no_store (once), a call that
fails is mirror_failed (with the error): one line each, never a failed ingest; the walk repairs the mirror (P9.3).
MANAGED_MIRROR is off | rag_engine | vertex_search | both and is refused unless RESIDENCY=us: serverless corpora are
us-central1-only and data stores are global, so neither keeps the India story (D6). Figure and segment chunks stay
on the kit's own index (D4): a store receives text, and only text.
"""
from __future__ import annotations

import argparse
import json
import logging
import os
import re
import sys
import tempfile

log = logging.getLogger("documind.ingest")

MODES = ("off", "rag_engine", "vertex_search", "both")
CHUNK_SIZE, CHUNK_OVERLAP = 512, 100          # what 4.3's ManagedRAG.ingest() asks of RAG Engine
STRUCT_FIELDS = ("tenant_id", "doc_key", "source_uri", "kind", "title", "doc_type", "effective_from", "generation", "name")


def check_mode(mode: str | None, residency: str | None) -> str:
    """MANAGED_MIRROR against RESIDENCY, at startup: an unknown value and a mirror under any residency but `us` are
    refused with the reason, so a revision that would ship a tenant's text out of India never serves."""
    mode = (mode or "off").strip().lower()
    if mode not in MODES:
        raise ValueError(f"MANAGED_MIRROR={mode!r}: one of {'|'.join(MODES)}")
    if mode != "off" and (residency or "india") != "us":
        raise RuntimeError(f"MANAGED_MIRROR={mode} needs RESIDENCY=us: serverless RAG Engine corpora are us-central1-only and "
                           "Vertex AI Search data stores are global (4.3, 4.4), so neither keeps the India story. Set "
                           "MANAGED_MIRROR=off, or run the lane with RESIDENCY=us.")
    return mode


def store_id(tenant_id: str) -> str:
    """documind-{tenant}: the corpus's display name and the data store's id (both take [a-z0-9-])."""
    return "documind-" + re.sub(r"[^a-z0-9-]+", "-", tenant_id.lower()).strip("-")


def version_rows(db, tenant_id: str, doc_key: str) -> tuple[str, dict]:
    """A version as the kit's rows hold it: every current text chunk of the doc_key in the order the chunker minted
    them (the index after `#` in a worker id, the locator in a notebook id), joined by blank lines - plus the
    doc_type and effective_from the rows carry. What after_undo() mirrors and what the walk re-mirrors."""
    rows, meta = [], {}
    for snap in db.collection("chunks").where("tenant_id", "==", tenant_id).where("doc_key", "==", doc_key).stream():
        d = snap.to_dict() or {}
        if d.get("current") is False or d.get("kind", "text") != "text":
            continue
        tail = snap.id.rsplit("#", 1)[-1]
        rows.append((int(tail) if tail.isdigit() else 10 ** 9, tail, d.get("text") or ""))
        meta.setdefault("doc_type", d.get("doc_type"))
        meta.setdefault("effective_from", d.get("effective_from"))
    rows.sort()
    return "\\n\\n".join(t for _, _, t in rows if t.strip()), {k: v for k, v in meta.items() if v is not None}


class NoStore(Exception):
    """The tenant has no store of this kind. Stores are declared - managed.tf, make rag-corpus - never created here."""


class RagEngineStore:
    """A RAG Engine corpus per tenant (4.3): the version's text uploaded as one RagFile named by its doc_key, chunked
    the lesson's way (512 / 100). `rag` is vertexai.rag, injectable for the gate."""
    name = "rag_engine"

    def __init__(self, project: str, location: str = "us-central1", embedding_model: str = "text-embedding-005", rag=None):
        self.project, self.location, self.embedding_model = project, location, embedding_model
        self._rag = rag
        self._corpora: dict[str, str | None] = {}

    def rag(self):
        if self._rag is None:
            import vertexai
            from vertexai import rag
            vertexai.init(project=self.project, location=self.location)   # corpora are regional; only generation is global
            self._rag = rag
        return self._rag

    def corpus(self, tenant_id: str) -> str:
        if tenant_id not in self._corpora:
            want = store_id(tenant_id)
            self._corpora[tenant_id] = next((c.name for c in self.rag().list_corpora() if c.display_name == want), None)
        name = self._corpora[tenant_id]
        if not name:
            raise NoStore(f"no RAG Engine corpus {store_id(tenant_id)!r} in {self.location}: make rag-corpus TENANT={tenant_id}")
        return name

    def create(self, tenant_id: str) -> str:
        """Once per tenant, by display name - create_corpus does not de-duplicate (4.3's rule), and a second corpus
        would bill for ever. The kit's one declared embedding model, so the store and the rows agree."""
        rag = self.rag()
        want = store_id(tenant_id)
        for c in rag.list_corpora():
            if c.display_name == want:
                self._corpora[tenant_id] = c.name
                return c.name
        emb = rag.RagEmbeddingModelConfig(vertex_prediction_endpoint=rag.VertexPredictionEndpoint(
            publisher_model=f"publishers/google/models/{self.embedding_model}"))
        c = rag.create_corpus(display_name=want, description=f"DocuMind managed mirror, tenant {tenant_id} (services/ingest/managed.py)",
                              backend_config=rag.RagVectorDbConfig(rag_embedding_model_config=emb))
        self._corpora[tenant_id] = c.name
        return c.name

    def _files(self, corpus_name: str, doc_key: str) -> list:
        return [f for f in self.rag().list_files(corpus_name) if f.display_name == doc_key]

    def upsert(self, tenant_id: str, doc_key: str, source_uri: str, text: str, meta: dict) -> str:
        rag = self.rag()
        corpus_name = self.corpus(tenant_id)
        for f in self._files(corpus_name, doc_key):            # the same version again: one file per doc_key
            rag.delete_file(name=f.name, corpus_name=corpus_name)
        with tempfile.TemporaryDirectory() as d:
            path = os.path.join(d, f"{doc_key}.txt")
            with open(path, "w", encoding="utf-8") as fh:
                fh.write(text)
            f = rag.upload_file(corpus_name=corpus_name, path=path, display_name=doc_key, description=source_uri,
                                transformation_config=rag.TransformationConfig(
                                    chunking_config=rag.ChunkingConfig(chunk_size=CHUNK_SIZE, chunk_overlap=CHUNK_OVERLAP)))
        return f.name

    def delete(self, tenant_id: str, doc_key: str) -> int:
        rag = self.rag()
        corpus_name = self.corpus(tenant_id)
        gone = 0
        for f in self._files(corpus_name, doc_key):
            rag.delete_file(name=f.name, corpus_name=corpus_name)
            gone += 1
        return gone

    def listing(self, tenant_id: str) -> dict:
        corpus_name = self.corpus(tenant_id)
        return {f.display_name: {"name": f.name, "source_uri": f.description} for f in self.rag().list_files(corpus_name)}


class VertexSearchStore:
    """A Vertex AI Search data store per tenant (4.4, managed.tf): the version's text as an object in the audit bucket
    (which has no notification, so nothing ingests it; Document.content.raw_bytes has a 1 MB ceiling, an object has
    none) imported inline as one Document whose id is the doc_key and whose structData is the schema's fields.
    `clients` (de, docs, ds, gcs) is injectable for the gate."""
    name = "vertex_search"

    def __init__(self, project: str, location: str = "global", bucket: str = "", clients=None):
        self.project, self.location, self.bucket = project, location, bucket
        self._c = clients
        self._exists: dict[str, bool] = {}

    def clients(self):
        if self._c is None:
            from types import SimpleNamespace
            from google.cloud import discoveryengine_v1 as de
            from google.cloud import storage
            self._c = SimpleNamespace(de=de, docs=de.DocumentServiceClient(), ds=de.DataStoreServiceClient(),
                                      gcs=storage.Client(project=self.project))
        return self._c

    def data_store(self, tenant_id: str) -> str:
        return (f"projects/{self.project}/locations/{self.location}/collections/default_collection"
                f"/dataStores/{store_id(tenant_id)}")

    def branch(self, tenant_id: str) -> str:
        return f"{self.data_store(tenant_id)}/branches/default_branch"

    def ensure(self, tenant_id: str) -> None:
        c = self.clients()
        if tenant_id not in self._exists:
            from google.api_core.exceptions import NotFound
            try:
                c.ds.get_data_store(name=self.data_store(tenant_id))
                self._exists[tenant_id] = True
            except NotFound:
                self._exists[tenant_id] = False
        if not self._exists[tenant_id]:
            raise NoStore(f"no Vertex AI Search data store {store_id(tenant_id)!r} in {self.location}: "
                          f"MANAGED_SEARCH=true make plan / make up declares one per tenant (managed.tf)")

    def object_name(self, tenant_id: str, doc_key: str) -> str:
        return f"search/{tenant_id}/{doc_key}.txt"

    def upsert(self, tenant_id: str, doc_key: str, source_uri: str, text: str, meta: dict) -> str:
        c = self.clients()
        self.ensure(tenant_id)
        if not self.bucket:
            raise RuntimeError("AUDIT_BUCKET is not set: the mirror keeps the version's text there")
        obj = self.object_name(tenant_id, doc_key)
        c.gcs.bucket(self.bucket).blob(obj).upload_from_string(text, content_type="text/plain; charset=utf-8")
        struct = {"tenant_id": tenant_id, "doc_key": doc_key, "source_uri": source_uri, "kind": "text",
                  "title": source_uri.rsplit("/", 1)[-1], **{k: v for k, v in meta.items() if k in STRUCT_FIELDS and v is not None}}
        doc = c.de.Document(id=doc_key, struct_data=struct,
                            content=c.de.Document.Content(uri=f"gs://{self.bucket}/{obj}", mime_type="text/plain"))
        op = c.docs.import_documents(request=c.de.ImportDocumentsRequest(
            parent=self.branch(tenant_id), inline_source=c.de.ImportDocumentsRequest.InlineSource(documents=[doc]),
            reconciliation_mode=c.de.ImportDocumentsRequest.ReconciliationMode.INCREMENTAL))
        return getattr(getattr(op, "operation", None), "name", "") or "import requested"   # an LRO: minutes; the walk verifies

    def delete(self, tenant_id: str, doc_key: str) -> int:
        c = self.clients()
        self.ensure(tenant_id)
        from google.api_core.exceptions import NotFound
        gone = 0
        try:
            c.docs.delete_document(name=f"{self.branch(tenant_id)}/documents/{doc_key}")
            gone = 1
        except NotFound:
            pass
        if self.bucket:
            try:
                c.gcs.bucket(self.bucket).blob(self.object_name(tenant_id, doc_key)).delete()
            except NotFound:
                pass
        return gone

    def listing(self, tenant_id: str) -> dict:
        c = self.clients()
        self.ensure(tenant_id)
        out = {}
        for doc in c.docs.list_documents(parent=self.branch(tenant_id)):
            sd = (c.de.Document.to_dict(doc).get("struct_data") or {}) if hasattr(c.de.Document, "to_dict") else dict(doc.struct_data or {})
            out[doc.id] = {"name": doc.name, "source_uri": sd.get("source_uri")}
        return out


class Mirror:
    """The worker's, the batch job's and the walk's one door to the stores: every call is one log line per store
    and raises nothing - an ingest is never failed by its mirror."""

    def __init__(self, db, stores: list, mode: str = "off"):
        self.db, self.stores, self.mode = db, list(stores), mode
        self._said: set[tuple[str, str]] = set()

    @classmethod
    def from_env(cls, db, mode: str | None = None, residency: str | None = None) -> "Mirror":
        mode = check_mode(os.environ.get("MANAGED_MIRROR", "off") if mode is None else mode,
                          os.environ.get("RESIDENCY", "india") if residency is None else residency)
        stores: list = []
        if mode in ("rag_engine", "both"):
            stores.append(RagEngineStore(os.environ.get("GOOGLE_CLOUD_PROJECT", ""), os.environ.get("RAG_LOCATION", "us-central1"),
                                         os.environ.get("EMBEDDING_MODEL", "text-embedding-005")))
        if mode in ("vertex_search", "both"):
            stores.append(VertexSearchStore(os.environ.get("GOOGLE_CLOUD_PROJECT", ""), os.environ.get("SEARCH_LOCATION", "global"),
                                            os.environ.get("AUDIT_BUCKET", "")))
        return cls(db, stores, mode)

    @property
    def active(self) -> bool:
        return bool(self.stores)

    def _each(self, op: str, tenant_id: str, doc_key: str, fn) -> None:
        for store in self.stores:
            try:
                result = fn(store)
            except NoStore as e:
                if (store.name, tenant_id) not in self._said:
                    self._said.add((store.name, tenant_id))
                    log.warning(json.dumps({"event": "mirror_no_store", "store": store.name, "tenant": tenant_id,
                                            "doc_key": doc_key, "op": op, "hint": str(e)}))
                continue
            except Exception as e:  # noqa: BLE001 - the mirror never fails the ingest; the walk repairs it
                log.warning(json.dumps({"event": "mirror_failed", "store": store.name, "tenant": tenant_id, "doc_key": doc_key,
                                        "op": op, "error": f"{type(e).__name__}: {e}"[:300]}))
                continue
            log.info(json.dumps({"event": "mirror_ok", "store": store.name, "tenant": tenant_id, "doc_key": doc_key,
                                 "op": op, "result": str(result)[:200]}))

    def upsert(self, tenant_id: str, doc_key: str, source_uri: str, text: str, meta: dict | None = None) -> None:
        if not self.stores:
            return
        if not text or not text.strip():
            log.info(json.dumps({"event": "mirror_skipped", "tenant": tenant_id, "doc_key": doc_key,
                                 "why": "no text: a media version stays on the kit's own index"}))
            return
        meta = {k: v for k, v in (meta or {}).items() if v is not None}
        self._each("upsert", tenant_id, doc_key, lambda s: s.upsert(tenant_id, doc_key, source_uri, text, meta))

    def retired(self, tenant_id: str, doc_keys, why: str = "retired") -> None:
        for doc_key in doc_keys:
            self._each(f"delete:{why}", tenant_id, doc_key, lambda s, k=doc_key: s.delete(tenant_id, k))

    def after_swap(self, doc, text: str, gone: dict, meta: dict | None = None) -> None:
        """swap_versions() just made `doc` current: the store gets its text, and the versions the swap retired leave."""
        if not self.stores:
            return
        self.upsert(doc.tenant_id, doc.doc_key, doc.gcs_uri, text,
                    {"doc_type": doc.doc_type, "effective_from": doc.effective_from, **(meta or {})})
        self.retired(doc.tenant_id, [k for k in gone.get("retired_doc_keys", []) if k != doc.doc_key], "superseded")

    def after_undo(self, tenant_id: str, gcs_uri: str, doc_key: str, gone: dict, meta: dict | None = None) -> None:
        """The undo made `doc_key` current again: its text comes off its own rows; the newer version leaves."""
        if not self.stores:
            return
        text, rows_meta = version_rows(self.db, tenant_id, doc_key)
        self.upsert(tenant_id, doc_key, gcs_uri, text, {**rows_meta, **(meta or {})})
        self.retired(tenant_id, [k for k in gone.get("retired_doc_keys", []) if k != doc_key], "superseded")


def status(db, stores: list, tenant_only: str | None = None) -> list[dict]:
    """Each tenant the ledger knows, each store: what the store holds against the ledger's current versions."""
    current: dict[str, set] = {}
    for snap in db.collection("sources").stream():
        row = snap.to_dict() or {}
        t = row.get("tenant_id")
        if not t or (tenant_only and t != tenant_only):
            continue
        current.setdefault(t, set())
        if row.get("status") == "indexed" and row.get("doc_key"):
            current[t].add(row["doc_key"])
    if tenant_only and tenant_only not in current:
        current[tenant_only] = set()
    out = []
    for tenant in sorted(current):
        for store in stores:
            line = {"tenant": tenant, "store": store.name, "ledger_current": len(current[tenant])}
            try:
                held = store.listing(tenant)
            except NoStore as e:
                out.append({**line, "status": "no store", "hint": str(e)})
                continue
            except Exception as e:  # noqa: BLE001
                out.append({**line, "status": "error", "error": f"{type(e).__name__}: {e}"[:200]})
                continue
            missing = sorted(current[tenant] - set(held))
            orphans = sorted(set(held) - current[tenant])
            out.append({**line, "held": len(held), "missing": len(missing), "orphans": len(orphans),
                        "status": "in sync" if not missing and not orphans else "drift",
                        "missing_doc_keys": missing[:5], "orphan_doc_keys": orphans[:5]})
    return out


def main() -> int:
    ap = argparse.ArgumentParser(description=__doc__.splitlines()[0])
    ap.add_argument("--project", default=os.environ.get("GOOGLE_CLOUD_PROJECT"))
    ap.add_argument("--tenant")
    ap.add_argument("--status", action="store_true", help="each tenant, each store, against the ledger")
    ap.add_argument("--create-corpus", action="store_true", help="the tenant's RAG Engine corpus, once (needs --tenant)")
    ap.add_argument("--mode", default=os.environ.get("MANAGED_MIRROR") or "both",
                    help="which stores --status reads: rag_engine | vertex_search | both")
    args = ap.parse_args()
    logging.basicConfig(level=logging.INFO, format="%(message)s", stream=sys.stdout)
    if not args.project:
        ap.error("--project is required")
    if args.create_corpus:
        if not args.tenant:
            ap.error("--create-corpus needs --tenant")
        name = RagEngineStore(args.project, os.environ.get("RAG_LOCATION", "us-central1"),
                              os.environ.get("EMBEDDING_MODEL", "text-embedding-005")).create(args.tenant)
        print(json.dumps({"event": "rag_corpus_ready", "tenant": args.tenant, "corpus": name,
                          "note": "one corpus per tenant, found by name; RAG Engine storage bills while it exists - make down names it"}))
        return 0
    if args.status:
        from google.cloud import firestore
        db = firestore.Client(project=args.project)
        mode = check_mode(args.mode, os.environ.get("RESIDENCY", "us"))
        os.environ.setdefault("GOOGLE_CLOUD_PROJECT", args.project)
        mirror = Mirror.from_env(db, mode=mode, residency="us")
        for line in status(db, mirror.stores, args.tenant):
            print(json.dumps(line))
        return 0
    ap.error("one of --status, --create-corpus")
    return 2


if __name__ == "__main__":
    sys.exit(main())
'''

with open('managed.py', 'w') as f: f.write(MANAGED_PY)
print('wrote managed.py')


In [ ]:
VECTOR_TF = '''
# The STREAM_UPDATE index, provisioned for the first time in this lesson.
# Modules 4 and 12.2 have been querying an index that the course never created -
# this is it.
resource "google_vertex_ai_index" "documind" {
  count = local.full ? 1 : 0   # the full profile only (variables.tf)
  region       = var.region
  display_name = "documind-chunks"
  description  = "DocuMind chunk embeddings, 768-d, streaming upserts"

  metadata {
    contents_delta_uri = "gs://${google_storage_bucket.uploads.name}/index-delta"
    config {
      dimensions                  = 768
      approximate_neighbors_count = 150
      distance_measure_type       = "DOT_PRODUCT_DISTANCE"
      algorithm_config {
        tree_ah_config {
          leaf_node_embedding_count    = 500
          leaf_nodes_to_search_percent = 7
        }
      }
    }
  }

  # STREAM_UPDATE, not BATCH_UPDATE. With BATCH_UPDATE the worker's
  # upsert_datapoints call is accepted and applied at the next batch job, so an
  # uploaded document is simply absent for hours and nothing reports an error.
  index_update_method = "STREAM_UPDATE"
}

resource "google_vertex_ai_index_endpoint" "documind" {
  count = local.full ? 1 : 0   # the full profile only (variables.tf)
  region                  = var.region
  display_name            = "documind-endpoint"
  public_endpoint_enabled = true
}

# An index and an endpoint are two things; neither of them serves a query. The
# DEPLOYED index is the third, and it is the one that costs money per hour - which
# is why it is easy to leave out of the terraform and then wonder why
# find_neighbors returns nothing against an index that plainly exists.
resource "google_vertex_ai_index_endpoint_deployed_index" "documind" {
  count = local.full ? 1 : 0   # the full profile only (variables.tf)
  index_endpoint    = google_vertex_ai_index_endpoint.documind[0].id
  index             = google_vertex_ai_index.documind[0].id
  deployed_index_id = "documind_chunks_v1"
  display_name      = "documind-chunks-v1"

  # One small replica. This is the line to raise for a live cohort and the line
  # to drop to zero afterwards - see the scale-down runbook in deploy/README.
  dedicated_resources {
    machine_spec { machine_type = "e2-standard-2" }
    min_replica_count = 1
    max_replica_count = 1
  }
}

# These two outputs are the whole contract with rag-api/config.py, which reads
# them as VECTOR_INDEX_ENDPOINT and VECTOR_DEPLOYED_INDEX_ID. The endpoint output
# is the RESOURCE NAME, not the public domain: retriever.py passes it straight to
# aiplatform.MatchingEngineIndexEndpoint(), which wants the name.
output "vector_index_endpoint" {
  description = "VECTOR_INDEX_ENDPOINT for rag-api"
  value       = one(google_vertex_ai_index_endpoint.documind[*].id)
}

output "vector_deployed_index_id" {
  description = "VECTOR_DEPLOYED_INDEX_ID for rag-api"
  value       = one(google_vertex_ai_index_endpoint_deployed_index.documind[*].deployed_index_id)
}

output "vector_index_name" {
  description = "VECTOR_INDEX_NAME for the ingest worker's upsert_datapoints"
  value       = one(google_vertex_ai_index.documind[*].id)
}
'''

with open('vector.tf', 'w') as f: f.write(VECTOR_TF)
print('wrote vector.tf')


In [ ]:
DOCAI_TF = '''
# One processor, chosen by residency - the terraform half of parser.py's PROCESSORS map.
# Keep the two in step: a type added here without a row there is a processor nothing calls.
locals {
  docai = {
    india = { location = "asia-south1", type = "OCR_PROCESSOR" }
    us    = { location = "us", type = "LAYOUT_PARSER_PROCESSOR" }
  }
  docai_cfg = local.docai[var.residency]
}

resource "google_document_ai_processor" "documind" {
  project      = var.project_id
  location     = local.docai_cfg.location
  display_name = "documind-parser"
  type         = local.docai_cfg.type
}

# parser.py builds the processor path itself, so it needs the bare id and the
# location separately - not the full resource name.
output "docai_processor_id" {
  description = "DOCAI_PROCESSOR_ID for the ingest worker"
  value       = element(split("/", google_document_ai_processor.documind.id), 5)
}

output "docai_location" {
  description = "matches RESIDENCY in services/ingest/parser.py"
  value       = local.docai_cfg.location
}
'''

with open('docai.tf', 'w') as f: f.write(DOCAI_TF)
print('wrote docai.tf')
print()
print("residency -> processor, as terraform will resolve it:")
for res, cfg in {"india": ("asia-south1", "OCR_PROCESSOR"),
                 "us": ("us", "LAYOUT_PARSER_PROCESSOR")}.items():
    print(f"  var.residency = {res:6} -> {cfg[1]:24} in {cfg[0]}")
print()
print("The Layout Parser gives structure - headings, tables, reading order - and runs in")
print("`us` only. An India-resident tenant gets Enterprise OCR, which returns text and")
print("positions but no layout. 12.5's chunker has to cope with both.")


In [ ]:
FIRESTORE_INDEXES_TF = '''
# Two vector indexes, both 768-d to match text-embedding-005.

# chunks: the chaos fallback for retriever.py. Written by indexer.py on every
# ingest; queried only when Vector Search is down.
resource "google_firestore_index" "chunks_vector" {
  project     = var.project_id
  database    = google_firestore_database.main.name
  collection  = "chunks"
  query_scope = "COLLECTION"

  # Equality filter FIRST, vector field LAST. Firestore will not accept the
  # reverse, and the error message points at an index name that looks correct.
  fields {
    field_path = "tenant_id"
    order      = "ASCENDING"
  }
  # Firestore records the document key between the ordered fields and the vector field, and
  # reports it back in that position. Declared here the same way, the provider reads the
  # index it wrote. Left out, every apply read a three-field index against a two-field
  # definition, planned a replacement, and the asynchronous deletion raced the re-creation
  # into a 409 - the first live applies (the first live run, 6 September 2026) looped on exactly that.
  fields {
    field_path = "__name__"
    order      = "ASCENDING"
  }
  fields {
    field_path = "embedding"
    vector_config {
      dimension = 768
      flat {}
    }
  }
}

# chunks, current only: the ledger's promise (12.5, 11 September 2026). A SECOND index, not a change to the
# first - a changed vector index is destroyed and re-created, and retrieval would be refused while it builds;
# this one builds beside the first, and RETRIEVAL_CURRENT_ONLY=on on the API is what starts using it, after
# `make backfill-current` has stamped the chunks written before the ledger.
resource "google_firestore_index" "chunks_current_vector" {
  project     = var.project_id
  database    = google_firestore_database.main.name
  collection  = "chunks"
  query_scope = "COLLECTION"

  fields {
    field_path = "tenant_id"
    order      = "ASCENDING"
  }
  fields {
    field_path = "current"
    order      = "ASCENDING"
  }
  fields {
    field_path = "__name__"
    order      = "ASCENDING"
  }
  fields {
    field_path = "embedding"
    vector_config {
      dimension = 768
      flat {}
    }
  }
}

# The API's filters on the Firestore path (12 September 2026, retriever.py's _firestore_fallback applies
# doc_type and kind the way the Vector Search restricts do): Firestore refuses a vector query whose
# equality filters have no matching index rather than degrade, so each combination the API can send has
# one - with and without the ledger's `current`. Four indexes, built beside the two above.
locals {
  chunks_filter_indexes = {
    doc_type         = ["tenant_id", "doc_type"]
    current_doc_type = ["tenant_id", "current", "doc_type"]
    kind             = ["tenant_id", "kind"]
    current_kind     = ["tenant_id", "current", "kind"]
  }
}

resource "google_firestore_index" "chunks_filter_vector" {
  for_each    = local.chunks_filter_indexes
  project     = var.project_id
  database    = google_firestore_database.main.name
  collection  = "chunks"
  query_scope = "COLLECTION"

  dynamic "fields" {
    for_each = each.value
    content {
      field_path = fields.value
      order      = "ASCENDING"
    }
  }
  fields {
    field_path = "__name__"
    order      = "ASCENDING"
  }
  fields {
    field_path = "embedding"
    vector_config {
      dimension = 768
      flat {}
    }
  }
}

# answer_cache: the semantic cache from 12.6. Same shape, different collection -
# and the tenant_id filter is what stops one customer's answer reaching another.
resource "google_firestore_index" "answer_cache_vector" {
  project     = var.project_id
  database    = google_firestore_database.main.name
  collection  = "answer_cache"
  query_scope = "COLLECTION"

  fields {
    field_path = "tenant_id"
    order      = "ASCENDING"
  }
  # Firestore records the document key between the ordered fields and the vector field, and
  # reports it back in that position. Declared here the same way, the provider reads the
  # index it wrote. Left out, every apply read a three-field index against a two-field
  # definition, planned a replacement, and the asynchronous deletion raced the re-creation
  # into a 409 - the first live applies (the first live run, 6 September 2026) looped on exactly that.
  fields {
    field_path = "__name__"
    order      = "ASCENDING"
  }
  fields {
    field_path = "embedding"
    vector_config {
      dimension = 768
      flat {}
    }
  }
}

# Retention (12 September 2026, deploy/INDEXING.md). The ledger retires a chunk row with a flag and a stamp -
# expire_at = superseded_at + retention_days (variables.tf; the worker reads it as RETENTION_DAYS) - and this
# TTL policy is the ONLY thing that ever deletes one: the platform removes the row within about a day of the
# stamp, no account on the lane needs a delete, no cron runs one. The window is the audit window (a citation
# can still open what it quoted) and the undo window (reactivate clears the stamp) at once. A staged version
# the worker never swapped carries a one-day stamp and leaves the same way.
resource "google_firestore_field" "chunks_expire_at" {
  project    = var.project_id
  database   = google_firestore_database.main.name
  collection = "chunks"
  field      = "expire_at"

  ttl_config {}
}

# The answer cache (12.6, SEMANTIC_CACHE=on, 12 September 2026) stamps every entry with
# expire_at = created + SEMANTIC_CACHE_TTL_H, and this policy is what removes it; the lookup skips an
# expired entry the policy has not reached yet, so the ceiling holds either way.
resource "google_firestore_field" "answer_cache_expire_at" {
  project    = var.project_id
  database   = google_firestore_database.main.name
  collection = "answer_cache"
  field      = "expire_at"

  ttl_config {}
}

# The roster's reverse lookup (shared/tenancy.py tenant_for, lesson 12.8: "which tenant is
# this person on?") is a COLLECTION-GROUP query on members.email, across every tenant's
# members at once. Firestore indexes a single field at collection scope by default and
# refuses a collection-group query on it - the UI's first page was a FailedPrecondition
# on the first live sign-in (the first live run, 6 September 2026). This field override adds the
# collection-group index; the point lookup (is_member) reads by document id and needs none.
resource "google_firestore_field" "members_email" {
  project    = var.project_id
  database   = google_firestore_database.main.name
  collection = "members"
  field      = "email"
  index_config {
    indexes {
      order       = "ASCENDING"
      query_scope = "COLLECTION"
    }
    indexes {
      order       = "ASCENDING"
      query_scope = "COLLECTION_GROUP"
    }
  }
}
'''

with open('firestore_indexes.tf', 'w') as f: f.write(FIRESTORE_INDEXES_TF)
print('wrote firestore_indexes.tf')
print()
print('Index          collection     filter        vector field  dims')
print('-' * 62)
for name, coll in (('chunks_vector', 'chunks'), ('answer_cache_vector', 'answer_cache')):
    print(f'{name:14} {coll:14} tenant_id ==  embedding     768')
print()
print('Both are FLAT indexes. Firestore has no ANN tier to choose here: flat is')
print('exact and the collections are small enough per tenant that exact is fine.')


In [ ]:
EVENTARC_TF = '''
# GCS -> Pub/Sub -> Cloud Run, with a dead-letter topic.
#
# The bucket publishes its own object.finalized records (a Cloud Storage notification,
# payload JSON_API_V1) into the topic below, and the push subscription delivers them to the
# worker with an OIDC token and a floor of five attempts. The worker's IngestMessage is that
# record, field for field. An earlier draft declared an Eventarc trigger here instead: Eventarc
# delivers a CloudEvent straight to the service, around this subscription, its token, its retry
# ceiling and its DLQ - and the worker, which parses a Pub/Sub envelope, answered 400 to it.
data "google_project" "current" {}
data "google_storage_project_service_account" "gcs" {}

resource "google_pubsub_topic" "ingest" { name = "documind-ingest" }
resource "google_pubsub_topic" "ingest_dlq" { name = "documind-ingest-dlq" }

# Cloud Storage publishes as its own service agent. Without this grant the notification is
# created and nothing ever arrives - the first silent failure on this path.
resource "google_pubsub_topic_iam_member" "gcs_publishes" {
  topic  = google_pubsub_topic.ingest.id
  role   = "roles/pubsub.publisher"
  member = "serviceAccount:${data.google_storage_project_service_account.gcs.email_address}"
}

resource "google_storage_notification" "uploads" {
  bucket         = google_storage_bucket.uploads.name
  topic          = google_pubsub_topic.ingest.id
  payload_format = "JSON_API_V1"
  event_types    = ["OBJECT_FINALIZE"]
  depends_on     = [google_pubsub_topic_iam_member.gcs_publishes]
}

locals {
  # Cloud Run's deterministic URL: the service name and the project NUMBER, no hash to look
  # up after the first deploy. commands/lesson-12.2.sh builds SELF_URL the same way.
  ingest_url = "https://documind-ingest-${data.google_project.current.number}.${var.region}.run.app"
}

# Pub/Sub mints the push token AS the ingest service account, which takes this grant to
# Pub/Sub's own service agent - the second silent failure. The dead-letter hop needs the
# same agent to publish to the DLQ topic and to subscribe here.
resource "google_service_account_iam_member" "pubsub_mints_ingest_token" {
  service_account_id = google_service_account.ingest.name
  role               = "roles/iam.serviceAccountTokenCreator"
  member             = "serviceAccount:service-${data.google_project.current.number}@gcp-sa-pubsub.iam.gserviceaccount.com"
}

resource "google_pubsub_topic_iam_member" "dlq_publisher" {
  topic  = google_pubsub_topic.ingest_dlq.id
  role   = "roles/pubsub.publisher"
  member = "serviceAccount:service-${data.google_project.current.number}@gcp-sa-pubsub.iam.gserviceaccount.com"
}

resource "google_pubsub_subscription" "ingest_push" {
  name  = "documind-ingest-push"
  topic = google_pubsub_topic.ingest.name

  push_config {
    push_endpoint = local.ingest_url
    oidc_token {
      # The worker is --no-allow-unauthenticated. Pub/Sub mints an OIDC token
      # for this service account, so the endpoint is reachable by Pub/Sub and
      # by nothing else on the internet. commands/lesson-12.5.sh grants the
      # account run.invoker on the service once the service exists.
      service_account_email = google_service_account.ingest.email
    }
  }

  # At-least-once, with a ceiling: attempts with exponential backoff, then the message goes
  # to the DLQ instead of being retried for a week. Twelve, not five: a corpus upload lands
  # thirty objects at once, the worker takes one request per instance, and Cloud Run refuses
  # ("no available instance") while it cold-starts - the first live load sent a document
  # to the DLQ on five refusals that were never the document's fault.
  retry_policy {
    minimum_backoff = "10s"
    maximum_backoff = "600s"
  }
  dead_letter_policy {
    dead_letter_topic     = google_pubsub_topic.ingest_dlq.id
    max_delivery_attempts = 12
  }
  ack_deadline_seconds = 600
  depends_on           = [google_service_account_iam_member.pubsub_mints_ingest_token]
}

resource "google_pubsub_subscription_iam_member" "dlq_subscriber" {
  subscription = google_pubsub_subscription.ingest_push.name
  role         = "roles/pubsub.subscriber"
  member       = "serviceAccount:service-${data.google_project.current.number}@gcp-sa-pubsub.iam.gserviceaccount.com"
}

# Somewhere to read the poison from. The README's Tier B block pulls from it by name.
resource "google_pubsub_subscription" "ingest_dlq_sub" {
  name  = "ingest-dlq-sub"
  topic = google_pubsub_topic.ingest_dlq.name
}
'''

with open('eventarc.tf', 'w') as f: f.write(EVENTARC_TF)
print('wrote eventarc.tf')


In [ ]:
BATCH_TF = '''
# The batch lane's consumer (12.5, 13 September 2026): the job that indexes the documents the push path handed
# off - anything over MAX_INLINE_PAGES, which cannot finish inside a push request's 600 seconds. Declared here on
# the ingest image the lane already runs (var.reconcile_image: make batch-job passes
# REGION-docker.pkg.dev/PROJECT/documind/ingest:SHA), beside the IAM grant that lets the worker start it and an
# hourly schedule that drains whatever the worker could not start. Like reconcile.tf, one apply: BATCH_JOB=true on
# make plan / make up once an ingest image exists (the job cannot be created before its image is pushed), and
# `make batch-job` is an apply with the switch on. The worker learns the job's name through BATCH_JOB in its own
# environment (commands/lesson-12.5.sh) and starts it the moment it queues a document.
variable "batch_job" {
  type        = bool
  default     = false
  description = "declare and schedule documind-ingest-batch, the batch lane's consumer (needs an ingest image: make build first)"
}
locals {
  batch = var.batch_job && var.reconcile_image != ""
}
resource "google_cloud_run_v2_job" "ingest_batch" {
  count    = local.batch ? 1 : 0
  name     = "documind-ingest-batch"
  location = var.region
  template {
    task_count = 1
    template {
      service_account = google_service_account.ingest.email
      max_retries     = 0       # the claim carries the retry: a failed document says why, the next run does not retake it
      timeout         = "7200s" # a thousand-page Act at a second a page through Document AI, with room
      containers {
        image   = var.reconcile_image
        command = ["python"]
        args    = ["batch.py", "--project", var.project_id]
        # The worker's own environment (commands/lesson-12.5.sh), so index_document() runs the same way from a job.
        env {
          name  = "GOOGLE_CLOUD_PROJECT"
          value = var.project_id
        }
        env {
          name  = "REGION"
          value = var.region
        }
        env {
          name  = "AUDIT_BUCKET"
          value = "${var.project_id}-audit"
        }
        env {
          name  = "RESIDENCY"
          value = var.residency
        }
        env {
          name  = "DOCAI_PROCESSOR_ID"
          value = element(split("/", google_document_ai_processor.documind.id), 5)
        }
        env {
          name  = "RETENTION_DAYS"
          value = tostring(var.retention_days)
        }
        env {
          name  = "EMBEDDING_MODEL"
          value = var.embedding_model
        }
        env {
          name  = "EMBEDDING_VERSION"
          value = tostring(var.embedding_version)
        }
        env {
          name  = "VECTOR_INDEX_NAME"
          value = local.full ? one(google_vertex_ai_index.documind[*].id) : ""
        }
        env {
          name  = "BQ_CHUNK_TABLE"
          value = local.full ? "${var.project_id}.rag_data.chunk_source" : ""
        }
        env {
          name  = "MANAGED_MIRROR" # the managed mirror (P9.2): the same switch the worker runs with
          value = var.managed_mirror
        }
      }
    }
  }
  lifecycle {
    ignore_changes = [client, client_version]
  }
}
# The worker starts the job when it queues a document (main.py: _run_batch_job), and the schedule below runs it
# as the same account: run.invoker on the job is run.jobs.run.
resource "google_cloud_run_v2_job_iam_member" "ingest_batch_invoker" {
  count    = local.batch ? 1 : 0
  name     = google_cloud_run_v2_job.ingest_batch[0].name
  location = var.region
  role     = "roles/run.invoker"
  member   = "serviceAccount:${google_service_account.ingest.email}"
}
# Hourly, at a quarter past: the backstop for a document the worker queued but could not start the job for. An
# empty queue is one log line and no work.
resource "google_cloud_scheduler_job" "ingest_batch" {
  count       = local.batch ? 1 : 0
  name        = "documind-ingest-batch-hourly"
  description = "DocuMind: drain the batch lane (documents over the push path's page ceiling)"
  schedule    = "15 * * * *"
  time_zone   = "Asia/Kolkata"
  region      = var.region
  http_target {
    http_method = "POST"
    uri         = "https://run.googleapis.com/v2/projects/${var.project_id}/locations/${var.region}/jobs/${google_cloud_run_v2_job.ingest_batch[0].name}:run"
    oauth_token {
      service_account_email = google_service_account.ingest.email
    }
  }
  depends_on = [google_cloud_run_v2_job_iam_member.ingest_batch_invoker]
}
'''

with open('batch.tf', 'w') as f: f.write(BATCH_TF)
print('wrote batch.tf')


In [ ]:
MANAGED_TF = '''
# The managed stores (P9.1, 13 September 2026): what lessons 4.3 and 4.4 build, declared per tenant for the mirror
# (services/ingest/managed.py, MANAGED_MIRROR) and the rag_engine / vertex_search retrieval backends (P9.4, P9.5).
# One store per tenant - RAG Engine has no metadata filter and Vertex AI Search's needs a schema, so isolation is
# structural (the plan's D2). A Vertex AI Search data store is a Terraform resource and is declared here, in
# `global` (the only home a generic data store has), behind MANAGED_SEARCH=true. A RAG Engine corpus is not a
# Terraform resource: `make rag-corpus TENANT=` creates one idempotently by display name, in us-central1 (the
# only region serverless corpora have), after `make rag-engine-enable` has switched the project to serverless
# mode once (4.3's one-time prep). Neither store keeps the India story, which is why the worker refuses the
# mirror unless RESIDENCY=us (managed.py: check_mode).
variable "managed_search" {
  type        = bool
  default     = false
  description = "declare a Vertex AI Search data store per tenant (global) for the vertex_search mirror and backend"
}

variable "managed_mirror" {
  type        = string
  default     = "off"
  description = "MANAGED_MIRROR for the batch and reconcile jobs, the same value make deploy-services hands the worker: off | rag_engine | vertex_search | both"
  validation {
    condition     = contains(["off", "rag_engine", "vertex_search", "both"], var.managed_mirror)
    error_message = "managed_mirror must be off, rag_engine, vertex_search or both."
  }
}

variable "tenants" {
  type        = list(string)
  default     = ["acme", "zeta", "globex"] # the roster's three golden tenants (evals/golden.jsonl)
  description = "the tenants a managed store is declared for; a tenant not listed gets mirror_no_store, never a store created on the fly"
}

locals {
  managed_tenants = var.managed_search ? toset(var.tenants) : toset([])
}

# The data store: unstructured documents with metadata (CONTENT_REQUIRED, GENERIC, search). The worker imports
# each version's text as one Document whose id is the doc_key, so a re-issue replaces and a retirement deletes.
# No document_processing_config on purpose: the config is immutable, the default digital parsing serves text/plain,
# and P9.5 reads extractive segments, which need no chunk mode. The schema is ours, not the default: the fields
# the mirror writes, indexable so the API's doc_type / kind filters become filter expressions (P9.5).
resource "google_discovery_engine_data_store" "tenant" {
  for_each                     = local.managed_tenants
  location                     = "global"
  data_store_id                = "documind-${each.key}"
  display_name                 = "DocuMind ${each.key}"
  industry_vertical            = "GENERIC"
  content_config               = "CONTENT_REQUIRED"
  solution_types               = ["SOLUTION_TYPE_SEARCH"]
  create_advanced_site_search  = false
  skip_default_schema_creation = true
}

resource "google_discovery_engine_schema" "tenant" {
  for_each      = local.managed_tenants
  location      = google_discovery_engine_data_store.tenant[each.key].location
  data_store_id = google_discovery_engine_data_store.tenant[each.key].data_store_id
  schema_id     = "default_schema"
  json_schema = jsonencode({
    "$schema" = "https://json-schema.org/draft/2020-12/schema"
    type      = "object"
    properties = {
      title          = { type = "string", keyPropertyMapping = "title", retrievable = true }
      tenant_id      = { type = "string", indexable = true, retrievable = true }
      doc_key        = { type = "string", indexable = true, retrievable = true }
      source_uri     = { type = "string", indexable = true, retrievable = true }
      kind           = { type = "string", indexable = true, retrievable = true }
      doc_type       = { type = "string", indexable = true, retrievable = true, dynamicFacetable = true }
      effective_from = { type = "string", indexable = true, retrievable = true }
      generation     = { type = "string", retrievable = true }
      name           = { type = "string", retrievable = true }
    }
  })
}

output "managed_data_stores" {
  value       = { for t, ds in google_discovery_engine_data_store.tenant : t => ds.name }
  description = "the Vertex AI Search data store per tenant, when MANAGED_SEARCH=true"
}
'''

with open('managed.tf', 'w') as f: f.write(MANAGED_TF)
print('wrote managed.tf')


In [ ]:
REQUIREMENTS = '''
fastapi==0.141.1
uvicorn[standard]==0.52.4
gunicorn==26.2.0
pydantic==2.13.5
google-cloud-firestore==2.30.0
google-cloud-storage==3.13.1
google-cloud-documentai==3.15.0
pypdf==6.17.0
google-cloud-aiplatform==1.153.1
# the managed mirror (P9.2): a Vertex AI Search data store per tenant; RAG Engine rides aiplatform above
google-cloud-discoveryengine==0.13.11
google-genai==2.22.0
google-cloud-dlp==3.39.0
# 9.6: shared/pii.py re-encodes a page render under DLP's 0.5 MiB inspect limit before scanning its pixels.
Pillow==12.3.0
# The SQL lane (lesson 5.5, gap G9): one chunk_source row per indexed chunk, when BQ_CHUNK_TABLE is set.
google-cloud-bigquery==3.45.0
'''

with open('requirements.txt', 'w') as f: f.write(REQUIREMENTS)
print('wrote requirements.txt')


In [ ]:
DOCKERFILE = '''
# syntax=docker/dockerfile:1.7
FROM python:3.12-slim
RUN apt-get update && apt-get install -y --no-install-recommends tini ca-certificates && rm -rf /var/lib/apt/lists/*
RUN useradd --create-home --shell /bin/bash --uid 10001 app
WORKDIR /app
COPY services/ingest/requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt
COPY --chown=app:app shared/ ./shared/\nCOPY --chown=app:app services/ingest/ .
USER app
ENV PORT=8080 PYTHONUNBUFFERED=1
EXPOSE 8080
# CONCURRENCY 1, and one worker. The process holds a document claim while it
# parses and embeds; a second request on the same instance contends for CPU and
# stretches the ack window, and Pub/Sub redelivers whatever is not acked in
# time - which is how one slow instance turns one document into three
# deliveries and three claims.
ENTRYPOINT ["/usr/bin/tini", "--"]
CMD ["gunicorn", "-k", "uvicorn.workers.UvicornWorker", "-w", "1", "-b", "0.0.0.0:8080", "-t", "600", "--access-logfile", "-", "main:app"]
'''

with open('Dockerfile', 'w') as f: f.write(DOCKERFILE)
print('wrote Dockerfile')


In [ ]:
DEPLOY = '''
# The ingest worker. Reached by nothing but the push subscription in eventarc.tf, which
# delivers as documind-ingest-sa with an OIDC token - hence internal ingress, no
# unauthenticated calls, and the invoker grant to that account below. Concurrency 1 (see the
# Dockerfile), and a 600-second request ceiling to match the subscription's ack deadline:
# a hundred-page Act is a dozen Document AI calls and fits. Thirty instances, so a corpus
# upload of thirty objects is not a queue of cold-start refusals.
#
# RESIDENCY picks the processor the way docai.tf did (us: Layout Parser; india: OCR in Mumbai).
# VECTOR_INDEX_NAME and BQ_CHUNK_TABLE are empty in the lean profile, which switches the
# datapoint upsert and the BigQuery mirror off; the Firestore index is always written.
# 12 September 2026 (deploy/INDEXING.md): RETENTION_DAYS stamps expire_at on every retired row - the TTL policy's
# number, variables.tf's retention_days; EMBEDDING_MODEL / EMBEDDING_VERSION are the ONE declared embedding, stamped
# on every chunk row - the API embeds its queries with the same pair. make deploy-services passes all three.
# 13 September 2026: REGION and BATCH_JOB are the batch lane's consumer - the Cloud Run job batch.tf declares on this
# image (make batch-job), which the worker starts by name the moment it queues a document over MAX_INLINE_PAGES;
# empty until make deploy-services BATCH_JOB=true, and then the hourly schedule alone drains the queue.
# P9 (13 September 2026): MANAGED_MIRROR is the managed mirror - the version the worker swaps current copied into
# the tenant's RAG Engine corpus (4.3) and / or Vertex AI Search data store (4.4), managed.py; off on the lane,
# refused unless RESIDENCY=us. RAG_LOCATION is the corpora's region (serverless corpora: us-central1 only).
gcloud run deploy documind-ingest \\
  --image=${REGION:-us-central1}-docker.pkg.dev/$PROJECT/documind/ingest:$GIT_SHA \\
  --region=${REGION:-us-central1} --platform=managed \\
  --no-allow-unauthenticated \\
  --ingress=internal \\
  --memory=2Gi --cpu=2 --concurrency=1 --timeout=600 \\
  --min-instances=0 --max-instances=30 \\
  --execution-environment=gen2 \\
  --service-account=documind-ingest-sa@$PROJECT.iam.gserviceaccount.com \\
  --set-env-vars="^|^GOOGLE_CLOUD_PROJECT=$PROJECT|RESIDENCY=${RESIDENCY:-us}|DOCAI_PROCESSOR_ID=$DOCAI_PROCESSOR_ID|AUDIT_BUCKET=$PROJECT-audit|VECTOR_INDEX_NAME=$VECTOR_INDEX_NAME|BQ_CHUNK_TABLE=$BQ_CHUNK_TABLE|RETENTION_DAYS=${RETENTION_DAYS-30}|EMBEDDING_MODEL=${EMBEDDING_MODEL-text-embedding-005}|EMBEDDING_VERSION=${EMBEDDING_VERSION-1}|REGION=${REGION:-us-central1}|BATCH_JOB=${BATCH_JOB_NAME-}|MANAGED_MIRROR=${MANAGED_MIRROR-off}|RAG_LOCATION=${RAG_LOCATION-us-central1}"

# Pub/Sub calls the worker AS the ingest service account; the account has to be allowed in.
# The subscription already exists, pointing at this service's deterministic URL.
gcloud run services add-iam-policy-binding documind-ingest \\
  --region=${REGION:-us-central1} --project=$PROJECT \\
  --member="serviceAccount:documind-ingest-sa@$PROJECT.iam.gserviceaccount.com" \\
  --role=roles/run.invoker
'''
print(DEPLOY)
